#Real-World Retrieval-Augmented Generation (RAG) System - RAGBench

**Objective**:
Develop an efficient RAG pipeline capable of delivering accurate answers to the user queries.

# Stage 0: Prerequisites

Install Libraries

In [4]:
%%writefile requirements.txt
langchain
langchain-text-splitters
langchain-huggingface
pypdf
tiktoken
transformers
torch
sentence-transformers
groq
openai
gdown
python-dotenv
pymilvus[milvus_lite]
rank_bm25
llmlingua
gradio>=4.0
datasets
huggingface_hub
scikit-learn
matplotlib

Overwriting requirements.txt


In [5]:
!pip install -r requirements.txt -q --trusted-host pypi.org --trusted-host files.pythonhosted.org

In [6]:
# Utilities
import numpy as np
import pandas as pd
import re
import string
import json
import time
import random
import pprint
from getpass import getpass
import matplotlib.pyplot as plt
import tempfile
import uuid
from datetime import datetime

# Load environment variables
from dotenv import load_dotenv
import os
from getpass import getpass

# Vector store
#import faiss
# Milvus Lite (local, serverless)
from pymilvus import MilvusClient

# Groq LLM client
from groq import Groq

#Open router
from openai import OpenAI

#Hugging Face datasets
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi, hf_hub_download, list_repo_files

#Text Splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

#torch, sklearn and transformers
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import roc_auc_score

#HF
from huggingface_hub import HfFileSystem

# ── Clean start (remove stale partial downloads) ────────────────
import shutil

#BM25
from rank_bm25 import BM25Okapi

import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

Matplotlib is building the font cache; this may take a moment.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\koayalas.FAREAST\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\koayalas.FAREAST\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [7]:
# Choose your provider: "groq" or "openrouter"
LLM_PROVIDER = "openrouter"   # change to "groq" if preferred

In [8]:
# ── Cell 3: API Keys ───────────────────────────────────────────────────────────
import os
from getpass import getpass
from huggingface_hub import login

def get_secret_or_prompt(secret_name, prompt_text=None):
    """
    Try to read secret from Google Colab Secrets.
    If not available, ask user securely using getpass().
    """

    value = None

    # Try Colab Secrets first
    try:
        from google.colab import userdata
        value = userdata.get(secret_name)
    except Exception:
        value = None

    # Fallback to environment variable
    if not value:
        value = os.environ.get(secret_name)

    # Fallback to manual secure input
    if not value:
        prompt_text = prompt_text or f"Enter {secret_name}: "
        value = getpass(prompt_text)

    return value


# ── HuggingFace Token ─────────────────────────────────────────────────────────

HF_TOKEN = get_secret_or_prompt(
    "HF_TOKEN",
    "Enter HuggingFace Token: "
)

login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN

print("✅ HuggingFace token loaded and login completed")


# ── LLM Provider API Key ──────────────────────────────────────────────────────

if LLM_PROVIDER == "groq":

    GROQ_API_KEY = get_secret_or_prompt(
        "GROQ_API_KEY",
        "Enter GROQ API Key: "
    )

    OPENROUTER_API_KEY = None

    os.environ["GROQ_API_KEY"] = GROQ_API_KEY

    print("✅ GROQ API key loaded")

elif LLM_PROVIDER == "openrouter":

    OPENROUTER_API_KEY = get_secret_or_prompt(
        "OPENROUTER_API_KEY",
        "Enter OpenRouter API Key: "
    )

    GROQ_API_KEY = None

    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

    print("✅ OpenRouter API key loaded")

else:
    raise ValueError(f"Unknown LLM_PROVIDER: {LLM_PROVIDER}")


✅ HuggingFace token loaded and login completed
✅ OpenRouter API key loaded


Models

In [9]:
def get_llm_client():

    if LLM_PROVIDER == "groq":
        return Groq(
            api_key=GROQ_API_KEY,
            base_url="https://api.groq.com",
        )

    elif LLM_PROVIDER == "openrouter":
        return OpenAI(
            api_key=OPENROUTER_API_KEY,       # ✅ already loaded above
            base_url="https://openrouter.ai/api/v1"
        )

    else:
        raise ValueError(f"Unknown provider: {LLM_PROVIDER}")


# LLM client
llm_client = get_llm_client()

print(f"Provider : {LLM_PROVIDER}")

Provider : openrouter


In [ ]:
# ── HF CONFIG ──────────────────────────────────────────────────────
BUCKET_ID = "Phani555/IIITH-Cohort26-RAG-Batch37-storage"
BUCKET_PREFIX = f"hf://buckets/{BUCKET_ID}/milvus_dbs"
MILVUS_DIR = os.path.abspath("./milvus_store")  # local cache (pulled from HF)

fs = HfFileSystem()

# ── Config (use your repo) ─────────────────────────────────────
HF_REPO_ID = "Phani555/IIITH-Cohort26-RAG-Batch37-storage"
HF_REPO_TYPE = "dataset"
HF_FOLDER = "ablations"

# Stage 1 – Data Ingestion & Inspection


In [11]:
# load the full ragbench dataset
ragbench = {}
for dataset in ['covidqa', 'cuad', 'delucionqa', 'emanual', 'expertqa', 'finqa', 'hagrid', 'hotpotqa', 'msmarco', 'pubmedqa', 'tatqa', 'techqa']:
  ragbench[dataset] = load_dataset("rungalileo/ragbench", dataset)

In [12]:
def get_domain(dataset):
    if dataset in ("covidqa", "pubmedqa"):
        return "Bio_Medical"
    elif dataset in ("expertqa", "hagrid", "hotpotqa", "msmarco"):
        return "General_Knowledge"
    elif dataset in ("delucionqa", "emanual", "techqa"):
        return "Customer_Support"
    elif dataset in ("finqa", "tatqa"):
        return "Finance"
    else:
        return "Legal_Contracts"

ragbench_by_domain = {}
rows = []

for dataset, dataset_data in ragbench.items():
    domain = get_domain(dataset)

    ragbench_by_domain.setdefault(domain, {})
    ragbench_by_domain[domain][dataset] = dataset_data

    rows.append({
        "Domain": domain,
        "Dataset": dataset,
        "Train samples": len(dataset_data["train"]),
        "Test samples": len(dataset_data["test"]),
        "Validation samples": len(dataset_data["validation"])
    })

df_stats = pd.DataFrame(rows)

print(f"Total domains: {len(ragbench_by_domain)}")
print(f"Domains: {', '.join(ragbench_by_domain.keys())}\n")
print(df_stats.to_string(index=False))


Total domains: 5
Domains: Bio_Medical, Legal_Contracts, Customer_Support, General_Knowledge, Finance

           Domain    Dataset  Train samples  Test samples  Validation samples
      Bio_Medical    covidqa           1252           246                 267
  Legal_Contracts       cuad           1530           510                 510
 Customer_Support delucionqa           1460           184                 182
 Customer_Support    emanual           1054           132                 132
General_Knowledge   expertqa           1621           203                 203
          Finance      finqa          12502          2294                1766
General_Knowledge     hagrid           2892          1318                 322
General_Knowledge   hotpotqa           1883           390                 424
General_Knowledge    msmarco           1870           423                 397
      Bio_Medical   pubmedqa          19600          2450                2450
          Finance      tatqa          26

In [13]:
def print_sample_by_domain(ragbench_by_domain, domain, dataset_name, split="test", idx=0):
    example = ragbench_by_domain[domain][dataset_name][split][idx]

    print(f"Domain    : {domain}")
    print(f"Dataset   : {dataset_name}")
    print(f"Split     : {split}")
    print(f"Index     : {idx}")
    print("-" * 60)

    print("Question  :", example.get("question", ""))

    docs = example.get("documents", [])
    if docs:
        print("Document  :", docs[0][:100] + "...")
    else:
        print("Document  : N/A")

    print("Response  :", example.get("response", ""))

    print("\n--- Ground Truth Metrics ---")
    print("Relevance      :", example.get("relevance_score"))
    print("Utilization    :", example.get("utilization_score"))
    print("Completeness   :", example.get("completeness_score"))
    print("Adherence      :", example.get("adherence_score"))


In [14]:
print_sample_by_domain(
    ragbench_by_domain,
    domain="Finance",
    dataset_name="finqa",
    split="test",
    idx=0
)

Domain    : Finance
Dataset   : finqa
Split     : test
Index     : 0
------------------------------------------------------------
Question  : what is the rate of return in cadence design systems inc . of an investment from 2010 to 2011?
Document  : stockholder return performance graph the following graph compares the cumulative 5-year total stockh...
Response  : The rate of return in Cadence Design Systems Inc. from 2010 to 2011 is 37.9%. This is calculated by taking the value on 1/1/2011 (137.90) and subtracting the initial investment value on 1/2/2010 (100.00).

--- Ground Truth Metrics ---
Relevance      : 0.1111111111111111
Utilization    : 0.1111111111111111
Completeness   : 1.0
Adherence      : True


In [15]:
print_sample_by_domain(
    ragbench_by_domain,
    domain="Customer_Support",
    dataset_name="emanual",
    split="test",
    idx=0
)

Domain    : Customer_Support
Dataset   : emanual
Split     : test
Index     : 0
------------------------------------------------------------
Question  : I want to  enter into Ambient mode. How can I do that?
Document  : Changing the content and settings for Ambient Mode. When you press the button in Ambient Mode , the ...
Response  : To enter into Ambient Mode, you need to press the specified button on the remote control. In the provided context, it mentions that to enter Ambient Mode, you should press the button on the Samsung TV remote control.

--- Ground Truth Metrics ---
Relevance      : 0.02040816326530612
Utilization    : 0.04081632653061224
Completeness   : 1.0
Adherence      : True


In [16]:
print_sample_by_domain(
    ragbench_by_domain,
    domain="Bio_Medical",
    dataset_name="covidqa",
    split="test",
    idx=0
)

Domain    : Bio_Medical
Dataset   : covidqa
Split     : test
Index     : 0
------------------------------------------------------------
Question  : Which viruses may not cause prolonged inflammation due to strong induction of antiviral clearance?
Document  : Title: Type I Interferon Receptor Deficiency in Dendritic Cells Facilitates Systemic Murine Noroviru...
Response  : The viruses that may not cause prolonged inflammation due to strong induction of antiviral clearance are murine norovirus, human astrovirus, and murine cytomegalovirus.

--- Ground Truth Metrics ---
Relevance      : 0.4117647058823529
Utilization    : 0.17647058823529413
Completeness   : 0.42857142857142855
Adherence      : False


# Stage 2 – Chunking, Embeddings and Vector DB

In [17]:
#Choices
if LLM_PROVIDER == "openrouter":

    LLM_CHOICES = [

        # ── Generator Models ───────────────────────────────────
        "meta-llama/llama-3.1-8b-instruct",
        "openai/gpt-oss-120b",
        "meta-llama/llama-3.3-70b-instruct",

        # ── Judge / Reasoning Models ───────────────────────────
        "qwen/qwen3-32b",
        "moonshotai/kimi-k2-instruct",
        "deepseek/deepseek-r1",
        "openai/gpt-oss-20b",

        # ── Validation / Safety ────────────────────────────────
        "openai/gpt-oss-safeguard-20b",
    ]

else:

    LLM_CHOICES = [

        # ── Fast / Cheap ───────────────────────────────────────
        "llama-3.1-8b-instant",
        "gemma2-9b-it",

        # ── Main Generation ────────────────────────────────────
        "llama-3.3-70b-versatile",
        "qwen-qwq-32b",
        "mixtral-8x7b-32768",
        "deepseek-r1-distill-llama-70b",
    ]

In [18]:
# ── Tunable knobs ──────────────────────────────────────────────

#Generator - LLM
MODEL_NAME = LLM_CHOICES[0]
MODEL_NAME_BIG = LLM_CHOICES[3] #Judge LLM
HYDE_MODEL = LLM_CHOICES[0]

# Embedding Model
EMBEDDING_TYPE = "llm_embedder"    # bge_small | llm_embedder

#Chunking
chunk_size = 1000
chunk_overlap = 100

top_k = 3

# ── Feature flags ──────────────────────────────────────────────
ENABLE_HYBRID     = True    # dense + BM25
ENABLE_HYDE       = True    # hypothetical document expansion
ENABLE_RERANKING  = True    # MonoT5 reranker
RERANKER_TYPE = "monot5"     # monot5 | tilde

# ── Tunable knobs ──────────────────────────────────────────────
RETRIEVE_TOP_K    = 10      # initial candidates from each retriever
RERANK_TOP_K      = 3       # final docs after reranking
HYBRID_ALPHA      = 0.5     # 0=BM25 only, 1=dense only, 0.5=balanced
MONOT5_MODEL      = "castorini/monot5-base-msmarco-10k"

#Reciprocal Rank Fusion (RRF)
ENABLE_RRF = True

# Standard value used in literature
RRF_K = 60

# TILDE-style practical alternative
TILDE_MODEL = "BAAI/bge-reranker-base"

HYDE_LLM_MODEL = MODEL_NAME

# ── Prompt strategy feature flag ────────────────────────────────
PROMPT_STRATEGY = "short"   # "short" | "long" | "long_cot"

# ── Repacking feature flag ──────────────────────────────────────
ENABLE_REPACKING = True      # reorder for LLM attention
REPACK_STRATEGY  = "sides"   # "forward" | "reverse" | "sides"

# ── Feature flags ──────────────────────────────────────────────
ENABLE_SUMMARIZATION = True
SUMMARIZATION_TYPE   = "recomp"   # "recomp" | "longllmlingua"

# RECOMP config
RECOMP_TOP_K_SENTS      = 8
RECOMP_MIN_SCORE        = 0.10
RECOMP_GROUNDING_BOOST  = 0.15
RECOMP_MIN_KEEP_RATIO   = 0.60
RECOMP_KEEP_CRITICAL    = True

# LongLLMLingua config
LLMLINGUA_RATE     = 0.5          # compression rate (0.5 = 50% kept)
LLMLINGUA_MODEL    = "NousResearch/Llama-2-7b-hf"   # smaller works too

# QUERY Classification, Rewriting, decompostion
ENABLE_QUERY_REWRITING = False
ENABLE_QUERY_DECOMPOSITION = False
ENABLE_QUERY_CLASSIFICATION = False
MAX_SUBQUERIES = 3

In [19]:
EMBED_MODELS = {
    "bge_small": "BAAI/bge-small-en-v1.5",
    "llm_embedder": "BAAI/llm-embedder",
}

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {device}")
print(f"Embedding Type: {EMBEDDING_TYPE}")

# Validate embedding type
if EMBEDDING_TYPE not in EMBED_MODELS:
    raise ValueError(
        f"Unknown EMBEDDING_TYPE={EMBEDDING_TYPE}. "
        f"Valid values: {list(EMBED_MODELS.keys())}"
    )

# Load only the selected model
model_name = EMBED_MODELS[EMBEDDING_TYPE]

print(f"\nLoading: {model_name}")

embed_model = SentenceTransformer(
    model_name,
    device=device
)

print("\n✅ Model loaded successfully")

print(
    "Embedding dimension:",
    embed_model.get_sentence_embedding_dimension)

Device: cpu
Embedding Type: llm_embedder

Loading: BAAI/llm-embedder


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\koayalas.FAREAST\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\koayalas.FAREAST\.cache\huggingface\hub\models--BAAI--llm-embedder. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/28.8k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


✅ Model loaded successfully
Embedding dimension: <bound method SentenceTransformer.get_sentence_embedding_dimension of SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({})
)>


In [ ]:
MILVUS_DIR = os.path.join(os.path.abspath("./milvus_store"), "milvus_dbs")

EMBEDDING_TYPE = "llm_embedder"

# Examples:
# "default"
# "chunk_v2"
# "chunk_v3"
# "legal_exp"
# "bio_test"
INDEX_VERSION = "default"


def get_db_path(domain_name,
                embedding_type=None,
                index_version=None):
    """
    Return Milvus DB path for the specified embedding/index version.

    Examples:

    bge_small
    ---------
    ./milvus_store/milvus_dbs/Finance.db

    llm_embedder + default
    ----------------------
    ./milvus_store/milvus_dbs/llm_embedder/Finance.db

    llm_embedder + chunk_v2
    -----------------------
    ./milvus_store/milvus_dbs/llm_embedder_chunk_v2/Finance.db

    llm_embedder + chunk_v3
    -----------------------
    ./milvus_store/milvus_dbs/llm_embedder_chunk_v3/Finance.db
    """

    embedding_type = embedding_type or EMBEDDING_TYPE
    index_version = index_version or INDEX_VERSION

    # Original BGE indexes stay in root
    if embedding_type == "bge_small":

        db_dir = MILVUS_DIR

    # Any llm_embedder index version
    elif embedding_type == "llm_embedder":

        folder_name = (
            "llm_embedder"
            if index_version == "default"
            else f"llm_embedder_{index_version}"
        )

        db_dir = os.path.join(
            MILVUS_DIR,
            folder_name
        )

    else:

        raise ValueError(
            f"Unknown EMBEDDING_TYPE={embedding_type}"
        )

    os.makedirs(
        db_dir,
        exist_ok=True
    )

    return os.path.join(
        db_dir,
        f"{domain_name}.db"
    )

In [21]:
# ───────────────────────────────────────────────────────────────
# Generic Index Download with Fallback
#
# Example:
#
# INDEX_VERSION = "chunk_v3"
#
# Priority:
#   1. HF llm_embedder_chunk_v3/<domain>.db
#   2. HF llm_embedder/<domain>.db
#
# Destination:
#   /content/milvus_store/milvus_dbs/llm_embedder_chunk_v3/
# ───────────────────────────────────────────────────────────────

import os
import shutil

DOMAINS = list(ragbench_by_domain.keys())


def hf_path_exists(path):
    try:
        fs.ls(path)
        return True
    except Exception:
        return False


def get_index_folder(index_version=None):
    """
    default   -> llm_embedder
    chunk_v2  -> llm_embedder_chunk_v2
    chunk_v3  -> llm_embedder_chunk_v3
    xyz       -> llm_embedder_xyz
    """

    index_version = index_version or INDEX_VERSION

    return (
        "llm_embedder"
        if index_version == "default"
        else f"llm_embedder_{index_version}"
    )


def download_index_with_fallback(
    domains=None,
    embedding_type=None,
    index_version=None
):

    domains = domains or DOMAINS
    embedding_type = embedding_type or EMBEDDING_TYPE
    index_version = index_version or INDEX_VERSION

    # ----------------------------------------------------------
    # BGE-small
    # ----------------------------------------------------------

    if embedding_type == "bge_small":

        local_dir = MILVUS_DIR

        report = []

        for domain_name in domains:

            remote_path = (
                f"{BUCKET_PREFIX}/{domain_name}.db"
            )

            local_target = os.path.join(
                local_dir,
                f"{domain_name}.db"
            )

            try:

                if os.path.exists(local_target):

                    if os.path.isdir(local_target):
                        shutil.rmtree(local_target)
                    else:
                        os.remove(local_target)

                fs.get(
                    remote_path,
                    local_target,
                    recursive=True
                )

                report.append({
                    "domain": domain_name,
                    "status": "downloaded",
                    "source_type": "bge_small",
                    "local_target": local_target
                })

            except Exception as e:

                report.append({
                    "domain": domain_name,
                    "status": "failed",
                    "error": str(e),
                    "local_target": local_target
                })

        return report, local_dir

    # ----------------------------------------------------------
    # LLM Embedder
    # ----------------------------------------------------------

    elif embedding_type == "llm_embedder":

        preferred_folder = get_index_folder(
            index_version
        )

        local_dir = os.path.join(
            MILVUS_DIR,
            preferred_folder
        )

        os.makedirs(
            local_dir,
            exist_ok=True
        )

        report = []

        for domain_name in domains:

            preferred_remote = (
                f"{BUCKET_PREFIX}/"
                f"{preferred_folder}/"
                f"{domain_name}.db"
            )

            fallback_remote = (
                f"{BUCKET_PREFIX}/llm_embedder/"
                f"{domain_name}.db"
            )

            local_target = os.path.join(
                local_dir,
                f"{domain_name}.db"
            )

            # Remove old local copy
            if os.path.exists(local_target):

                if os.path.isdir(local_target):
                    shutil.rmtree(local_target)
                else:
                    os.remove(local_target)

            if hf_path_exists(preferred_remote):

                source = preferred_remote
                source_type = preferred_folder

            elif hf_path_exists(fallback_remote):

                source = fallback_remote
                source_type = "llm_embedder"

            else:

                report.append({
                    "domain": domain_name,
                    "status": "missing",
                    "source_type": None,
                    "local_target": local_target
                })

                continue

            try:

                fs.get(
                    source,
                    local_target,
                    recursive=True
                )

                report.append({
                    "domain": domain_name,
                    "status": "downloaded",
                    "source_type": source_type,
                    "local_target": local_target
                })

            except Exception as e:

                report.append({
                    "domain": domain_name,
                    "status": "failed",
                    "source_type": source_type,
                    "local_target": local_target,
                    "error": str(e)
                })

        return report, local_dir

    else:

        raise ValueError(
            f"Unknown EMBEDDING_TYPE={embedding_type}"
        )


# ───────────────────────────────────────────────────────────────
# Execute Download
# ───────────────────────────────────────────────────────────────

download_report, local_dir = download_index_with_fallback()

print("\n" + "=" * 70)
print(f"Embedding Type : {EMBEDDING_TYPE}")
print(f"Index Version  : {INDEX_VERSION}")
print("=" * 70)

for item in download_report:

    if item["status"] == "downloaded":

        print(
            f"✅ {item['domain']:20s} "
            f"-> {item['source_type']}"
        )

    elif item["status"] == "missing":

        print(
            f"❌ {item['domain']:20s} "
            f"-> MISSING"
        )

    else:

        print(
            f"⚠️ {item['domain']:20s} "
            f"-> FAILED"
        )

print("\nLocal Folder:")
print(local_dir)

(…)db/collections/bio_medical/manifest.json:   0%|          | 0.00/455 [00:00<?, ?B/s]

(…)llections/bio_medical/manifest.json.prev:   0%|          | 0.00/405 [00:00<?, ?B/s]

(…)_default/data/data_000001_013420.parquet:   0%|          | 0.00/44.9M [00:00<?, ?B/s]

(…)l.db/collections/bio_medical/schema.json:   0%|          | 0.00/667 [00:00<?, ?B/s]

(…)ollections/legal_contracts/manifest.json:   0%|          | 0.00/455 [00:00<?, ?B/s]

...

(…)nce.db/collections/finance/manifest.json:   0%|          | 0.00/455 [00:00<?, ?B/s]

(…)b/collections/finance/manifest.json.prev:   0%|          | 0.00/405 [00:00<?, ?B/s]

(…)_default/data/data_000001_029674.parquet:   0%|          | 0.00/99.1M [00:00<?, ?B/s]

(…)nance.db/collections/finance/schema.json:   0%|          | 0.00/663 [00:00<?, ?B/s]


Embedding Type : llm_embedder
Index Version  : default
✅ Bio_Medical          -> llm_embedder
✅ Legal_Contracts      -> llm_embedder
✅ Customer_Support     -> llm_embedder
✅ General_Knowledge    -> llm_embedder
✅ Finance              -> llm_embedder

Local Folder:
/content/milvus_store/milvus_dbs\llm_embedder


In [22]:
# ───────────────────────────────────────────────────────────────
# Chunk Configs
# Add future versions here only.
# ───────────────────────────────────────────────────────────────

CHUNK_CONFIGS = {
    "default": {
        "Customer_Support":  {"size": 1000, "overlap": 100},
        "Finance":           {"size": 1000, "overlap": 100},
        "General_Knowledge": {"size": 1000, "overlap": 100},
        "Legal_Contracts":   {"size": 1000, "overlap": 100},
        "Bio_Medical":       {"size": 1000, "overlap": 100},
    },

    "chunk_v2": {
        "Customer_Support":  {"size": 1000, "overlap": 100},
        "Finance":           {"size": 1000, "overlap": 100},
        "General_Knowledge": {"size": 1000, "overlap": 100},
        "Legal_Contracts":   {"size": 600,  "overlap": 120},
        "Bio_Medical":       {"size": 500,  "overlap": 100},
    },

    "chunk_v3": {
        "Customer_Support":  {"size": 1000, "overlap": 100},
        "Finance":           {"size": 1000, "overlap": 100},
        "General_Knowledge": {"size": 1000, "overlap": 100},
        "Legal_Contracts":   {"size": 500,  "overlap": 100},
        "Bio_Medical":       {"size": 400,  "overlap": 80},
    },

    "chunk_v4": {
        "Customer_Support":  {"size": 1000, "overlap": 100},
        "Finance":           {"size": 1000, "overlap": 100},
        "General_Knowledge": {"size": 1000, "overlap": 100},
        "Legal_Contracts":   {"size": 1000,  "overlap": 200},
        "Bio_Medical":       {"size": 1000,  "overlap": 200},
    },
}


# Optional:
# Use None to process all domains.
# Or restrict to specific domains:
REINDEX_DOMAINS = {"Legal_Contracts", "Bio_Medical"}
#REINDEX_DOMAINS = None


In [23]:
# ───────────────────────────────────────────────────────────────
# Initialize Existing Milvus Collections Only
# No indexing / embedding / insertion
# ───────────────────────────────────────────────────────────────

indexes = {}
all_splits = {}
milvus_clients = {}

for domain_name in ragbench_by_domain.keys():

    print(f"\n===== Loading domain: {domain_name} =====")

    db_path = get_db_path(
        domain_name,
        embedding_type=EMBEDDING_TYPE,
        index_version=INDEX_VERSION
    )

    collection_name = domain_name.lower()

    print(f"DB Path: {db_path}")

    if not os.path.exists(db_path):
        print(f"❌ DB not found: {db_path}")
        continue

    try:
        client = MilvusClient(db_path)

        if not client.has_collection(collection_name):
            print(
                f"❌ Collection '{collection_name}' "
                f"not found in {db_path}"
            )
            continue

        client.load_collection(
            collection_name=collection_name
        )

        stats = client.get_collection_stats(
            collection_name
        )

        row_count = int(
            stats.get("row_count", 0)
        )

        milvus_clients[domain_name] = client
        indexes[domain_name] = client

        print(f"✅ Loaded collection: {collection_name}")
        print(f"✅ Rows            : {row_count}")

    except Exception as e:
        print(
            f"❌ Failed loading "
            f"{domain_name}: {e}"
        )


===== Loading domain: Bio_Medical =====
DB Path: /content/milvus_store/milvus_dbs\llm_embedder\Bio_Medical.db
✅ Loaded collection: bio_medical
✅ Rows            : 0

===== Loading domain: Legal_Contracts =====
DB Path: /content/milvus_store/milvus_dbs\llm_embedder\Legal_Contracts.db
✅ Loaded collection: legal_contracts
✅ Rows            : 0

===== Loading domain: Customer_Support =====
DB Path: /content/milvus_store/milvus_dbs\llm_embedder\Customer_Support.db
✅ Loaded collection: customer_support
✅ Rows            : 9954

===== Loading domain: General_Knowledge =====
DB Path: /content/milvus_store/milvus_dbs\llm_embedder\General_Knowledge.db
✅ Loaded collection: general_knowledge
✅ Rows            : 0

===== Loading domain: Finance =====
DB Path: /content/milvus_store/milvus_dbs\llm_embedder\Finance.db
✅ Loaded collection: finance
✅ Rows            : 0


# Stage 3 – Retrieval

In [ ]:
# def retrieve(query, domain_name, embed_model=None, llm_client=None, top_k=3,rewritten_query=None):
#     """
#     Unified retrieval with feature flags:
#     - ENABLE_HYDE          → expand query with hypothetical answer
#     - ENABLE_HYBRID        → combine dense + BM25
#     - ENABLE_RERANKING     → MonoT5 / TILDE rerank
#     - ENABLE_SUMMARIZATION → RECOMP / LongLLMLingua
#     - ENABLE_REPACKING     → forward / reverse / sides

#     Pipeline order:
#     HyDE → Retrieve → Rerank → top_k → Summarize → Repack

#     Always returns:
#         list[dict] with at least "text" and "score" keys.
#     """

#     # ── Validate domain ────────────────────────────────────────
#     if domain_name not in milvus_clients:
#         raise ValueError(f"Domain '{domain_name}' not loaded.")

#     # ── Resolve dependencies ───────────────────────────────────
#     embed_model = embed_model or globals().get("embed_model")
#     llm_client  = llm_client  or globals().get("llm_client")

#     if embed_model is None:
#         raise ValueError("embed_model is required (not provided and not found in globals).")

#     # ── Determine retrieval depth ──────────────────────────────

#     fetch_k = RETRIEVE_TOP_K if (
#         ENABLE_HYBRID
#         or ENABLE_RERANKING
#         or ENABLE_SUMMARIZATION
#         or ENABLE_REPACKING
#     ) else top_k


#     # ── Step 1: HyDE query expansion ───────────────────────────

#     effective_query = rewritten_query if rewritten_query else query
#     search_query = effective_query

#     if ENABLE_HYDE and llm_client is not None:
#         try:
#             hyde_answer = generate_hyde(
#                 query=effective_query,
#                 llm_client=llm_client
#             )

#             if hyde_answer and str(hyde_answer).strip():
#                 search_query = f"{effective_query} {str(hyde_answer).strip()}"
#             else:
#                 print("HyDE returned empty response. Falling back to effective query.")
#                 search_query = effective_query

#         except Exception as e:
#             print(f"HyDE failed, falling back to effective query: {e}")
#             search_query = effective_query

#     # ── Step 2: Retrieval: hybrid or dense-only ────────────────
#     if ENABLE_HYBRID:
#         retrieved = hybrid_search(
#             query=search_query,
#             domain_name=domain_name,
#             embed_model=embed_model,
#             top_k=fetch_k,
#             alpha=HYBRID_ALPHA
#         )

#     else:
#         client = milvus_clients[domain_name]
#         collection_name = domain_name.lower()

#         if client.get_load_state(collection_name) != "Loaded":
#             client.load_collection(collection_name)

#         query_embed = embed_model.encode(
#             [search_query],
#             normalize_embeddings=True
#         ).astype("float32")

#         hits = client.search(
#             collection_name=collection_name,
#             data=query_embed.tolist(),
#             limit=fetch_k,
#             output_fields=["text"],
#             search_params={
#                 "metric_type": "IP",
#                 "params": {}
#             }
#         )

#         seen = set()
#         retrieved = []

#         for hit in hits[0]:
#             text = hit.entity.get("text", "")

#             if text and text not in seen:
#                 retrieved.append(
#                     {
#                         "text": text,
#                         "score": float(hit.distance)
#                     }
#                 )
#                 seen.add(text)

#     # ── Empty retrieval guard ──────────────────────────────────
#     if not retrieved:
#         return []

#     # ── Step 3: Unified reranking ──────────────────────────────
#     if ENABLE_RERANKING:
#         retrieved = rerank_documents(
#             query=effective_query,
#             documents=retrieved,
#             top_k=top_k
#         )

#     # ── Step 4: Final top_k ────────────────────────────────────
#     retrieved = retrieved[:top_k]

#     # ── Step 5: Summarization ──────────────────────────────────
#     if ENABLE_SUMMARIZATION:

#         summarized = summarize_docs(
#             query=effective_query,
#             docs=retrieved,
#             embed_model=embed_model,
#             llm_client=llm_client
#         )

#         # Preserve score / reranker information
#         scores_for_summary = [
#             d.get("rerank_score", d.get("score", 0.0))
#             for d in retrieved
#         ]

#         avg_score = (
#             float(np.mean(scores_for_summary))
#             if scores_for_summary else 1.0
#         )

#         max_rerank_score = (
#             float(max(scores_for_summary))
#             if scores_for_summary else avg_score
#         )

#         reranker_type = (
#             retrieved[0].get("reranker_type")
#             if retrieved and "reranker_type" in retrieved[0]
#             else None
#         )

#         retrieved = [
#             {
#                 "text": s,

#                 # summary-level score
#                 "score": avg_score,

#                 # preserve reranker metadata
#                 "rerank_score": max_rerank_score,
#                 "reranker_type": reranker_type,

#                 # summarization metadata
#                 "summarized": True,
#                 "summary_type": SUMMARIZATION_TYPE,
#             }
#             for s in summarized
#             if s and str(s).strip()
#         ]

#     # ── Step 6: Repacking ──────────────────────────────────────
#     if ENABLE_REPACKING:
#         retrieved = repack_documents(
#             retrieved,
#             strategy=REPACK_STRATEGY
#         )

#     return retrieved

In [ ]:
def retrieve(
    query,
    domain_name,
    embed_model=None,
    llm_client=None,
    top_k=3,
    rewritten_query=None
):
    """
    Unified retrieval with feature flags:
    - ENABLE_HYDE          → expand query with hypothetical answer
    - ENABLE_HYBRID        → combine dense + BM25 / RRF if enabled inside hybrid_search
    - ENABLE_RERANKING     → MonoT5 / TILDE rerank
    - ENABLE_SUMMARIZATION → RECOMP / LongLLMLingua
    - ENABLE_REPACKING     → forward / reverse / sides

    Pipeline order:
        HyDE → Retrieve → Rerank → top_k → Repack → Summarize

    Always returns:
        list[dict] with at least:
        - text
        - score
    """

    # ─────────────────────────────────
    # 0) Validate domain
    # ─────────────────────────────────
    if domain_name not in milvus_clients:
        raise ValueError(f"Domain '{domain_name}' not loaded in milvus_clients.")

    # ─────────────────────────────────
    # 1) Resolve dependencies
    # ─────────────────────────────────
    embed_model = embed_model or globals().get("embed_model")
    llm_client  = llm_client  or globals().get("llm_client")

    if embed_model is None:
        raise ValueError(
            "embed_model is required but was not provided and was not found in globals()."
        )

    # ─────────────────────────────────
    # 2) Determine retrieval depth
    # ─────────────────────────────────
    # Retrieve more only when a downstream stage benefits from wider candidates.
    # Repacking alone does not need deeper retrieval.
    fetch_k = RETRIEVE_TOP_K if (
        ENABLE_HYBRID
        or ENABLE_RERANKING
        or ENABLE_SUMMARIZATION
    ) else top_k

    fetch_k = max(fetch_k, top_k)

    # ─────────────────────────────────
    # 3) Resolve effective query
    # ─────────────────────────────────
    effective_query = rewritten_query if rewritten_query else query
    search_query = effective_query

    # ─────────────────────────────────
    # 4) HyDE query expansion
    # ─────────────────────────────────
    if ENABLE_HYDE:
        if llm_client is None:
            print("HyDE enabled but llm_client is missing. Falling back to effective query.")
        else:
            try:
                hyde_answer = generate_hyde(
                    query=effective_query,
                    llm_client=llm_client
                )

                if hyde_answer and str(hyde_answer).strip():
                    search_query = f"{effective_query} {str(hyde_answer).strip()}"
                else:
                    print("HyDE returned empty response. Falling back to effective query.")
                    search_query = effective_query

            except Exception as e:
                print(f"HyDE failed, falling back to effective query: {e}")
                search_query = effective_query

    # ─────────────────────────────────
    # 5) Retrieval: Hybrid or Dense-only
    # ─────────────────────────────────
    if ENABLE_HYBRID:

        retrieved = hybrid_search(
            query=search_query,
            domain_name=domain_name,
            embed_model=embed_model,
            top_k=fetch_k,
            alpha=HYBRID_ALPHA
        )

        # Add retrieval type if hybrid_search did not already add it
        for d in retrieved:
            if isinstance(d, dict):
                d.setdefault("retrieval_type", "hybrid")

    else:
        client = milvus_clients[domain_name]
        collection_name = domain_name.lower()

        # Safer collection loading for Milvus Lite / PyMilvus variations
        try:
            load_state = client.get_load_state(collection_name)

            if "Loaded" not in str(load_state):
                client.load_collection(collection_name)

        except Exception:
            try:
                client.load_collection(collection_name)
            except Exception:
                pass

        query_embed = embed_model.encode(
            [search_query],
            normalize_embeddings=True
        ).astype("float32")

        hits = client.search(
            collection_name=collection_name,
            data=query_embed.tolist(),
            limit=fetch_k,
            output_fields=["text"],
            search_params={
                "metric_type": "IP",
                "params": {}
            }
        )

        seen = set()
        retrieved = []

        # FIXED BUG HERE:
        # Old broken line was:
        # for hit in hitstext = hit.entity.get("text", "")
        for hit in hits[0]:
            text = hit.entity.get("text", "")

            if text and text not in seen:
                retrieved.append({
                    "text": text,
                    "score": float(hit.distance),
                    "retrieval_type": "dense"
                })
                seen.add(text)

    # ─────────────────────────────────
    # 6) Empty retrieval guard
    # ─────────────────────────────────
    if not retrieved:
        return []

    # ─────────────────────────────────
    # 7) Reranking
    # ─────────────────────────────────
    if ENABLE_RERANKING:
        retrieved = rerank_documents(
            query=effective_query,
            documents=retrieved,
            top_k=top_k
        )
    else:
        retrieved = retrieved[:top_k]

    # ─────────────────────────────────
    # 8) Repacking before summarization
    # ─────────────────────────────────
    if ENABLE_REPACKING:
        retrieved = repack_documents(
            retrieved,
            strategy=REPACK_STRATEGY
        )

    # ─────────────────────────────────
    # 9) Summarization
    # ─────────────────────────────────
    if ENABLE_SUMMARIZATION:

        original_retrieved = retrieved

        summarized = summarize_docs(
            query=effective_query,
            docs=retrieved,
            embed_model=embed_model,
            llm_client=llm_client
        )

        # If summarization fails or returns empty, preserve original retrieved docs
        if not summarized:
            print("Summarization returned empty output. Falling back to retrieved docs.")
            return original_retrieved

        scores_for_summary = [
            d.get("rerank_score", d.get("score", 0.0))
            for d in retrieved
            if isinstance(d, dict)
        ]

        avg_score = (
            float(np.mean(scores_for_summary))
            if scores_for_summary else 1.0
        )

        max_rerank_score = (
            float(max(scores_for_summary))
            if scores_for_summary else avg_score
        )

        reranker_type = (
            retrieved[0].get("reranker_type")
            if retrieved
            and isinstance(retrieved[0], dict)
            and "reranker_type" in retrieved[0]
            else None
        )

        retrieval_type = (
            retrieved[0].get("retrieval_type")
            if retrieved
            and isinstance(retrieved[0], dict)
            and "retrieval_type" in retrieved[0]
            else None
        )

        retrieved = [
            {
                "text": s,

                # Summary-level score
                "score": avg_score,

                # Preserve reranker metadata
                "rerank_score": max_rerank_score,
                "reranker_type": reranker_type,

                # Preserve retrieval metadata
                "retrieval_type": retrieval_type,

                # Summarization metadata
                "summarized": True,
                "summary_type": SUMMARIZATION_TYPE,
                "source_doc_count": len(scores_for_summary),
                "source_scores": scores_for_summary,
            }
            for s in summarized
            if s and str(s).strip()
        ]

        # Final safety fallback
        if not retrieved:
            print("Summarization produced only empty strings. Falling back to retrieved docs.")
            return original_retrieved

    return retrieved


print("Unified retrieve() function loaded ✅")

In [ ]:
def answer_question_with_classifier(
    question: str,
    domain_name: str,
    llm_client,
    embed_model=None,
    top_k: int = 3,
    rag_strategy: str = None
) -> dict:
    """
    Routes the query based on classification and answers the question.

    Supports:
    - query classification
    - query decomposition
    - query rewriting
    - RAG answering
    - direct LLM answering
    """

    rag_strategy = rag_strategy or globals().get("PROMPT_STRATEGY", "short")

    # ── Query Classification ─────────────────────────────────────

    if ENABLE_QUERY_CLASSIFICATION:

        classification = classify_query(
            question,
            domain_name=domain_name
        )

        print(
            f"Query Classification Enabled "
            f"-> {classification}"
        )

    else:

        classification = "RAG"

        print(
            "Query Classification Disabled "
            "-> Forcing RAG"
        )

    print("\n" + "=" * 80)
    print(f"Query          : {question}")
    print(f"Domain         : {domain_name}")
    print(f"Classification : {classification}")
    print("=" * 80)

    # Direct LLM route
    if classification == "LLM":
        answer = ask_llm(
            question,
            llm_client=llm_client
        )

        return {
            "source": "LLM",
            "question": question,
            "answer": answer,
            "context": None,
            "retrieved_docs": None,
            "subqueries": [question],
            "rewritten_queries": [question],
        }

    # RAG route
    if ENABLE_QUERY_DECOMPOSITION:
        subqueries = decompose_query(
            query=question,
            llm_client=llm_client,
            domain=domain_name,
            model=QUERY_DECOMPOSE_MODEL
        )
    else:
        subqueries = [question]

    print(f"Subqueries: {subqueries}")

    all_answers = []
    all_contexts = []
    all_retrieved_docs = []
    rewritten_queries = []

    for i, current_query in enumerate(subqueries, start=1):

        print("\n" + "-" * 80)
        print(f"Subquery {i}: {current_query}")

        if ENABLE_QUERY_REWRITING:
            rewritten_query = rewrite_query(
                query=current_query,
                domain_name=domain_name,
                llm_client=llm_client,
                model_name=QUERY_REWRITE_MODEL
            )
        else:
            rewritten_query = current_query

        rewritten_queries.append(rewritten_query)

        print(f"Rewritten : {rewritten_query}")

        retrieved_docs = retrieve(
            query=current_query,
            domain_name=domain_name,
            embed_model=embed_model,
            llm_client=llm_client,
            top_k=top_k,
            rewritten_query=rewritten_query
        )

        docs_text = [
            d.get("text", "") if isinstance(d, dict) else str(d)
            for d in retrieved_docs
        ]

        context = "\n\n".join(docs_text)

        answer = ask_rag(
            context=context,
            question=rewritten_query,
            llm_client=llm_client,
            strategy=rag_strategy
        )

        all_answers.append(
            f"{i}. {answer}"
        )

        all_contexts.append(context)
        all_retrieved_docs.extend(retrieved_docs)

    final_answer = "\n\n".join(all_answers)
    final_context = "\n\n".join(all_contexts)

    return {
        "source": "RAG",
        "question": question,
        "answer": final_answer,
        "context": final_context,
        "retrieved_docs": all_retrieved_docs,
        "subqueries": subqueries,
        "rewritten_queries": rewritten_queries,
    }

In [ ]:
''' FAISS
def retrieve(query, domain_name, top_k):
    if domain_name not in indexes:
        raise ValueError(f"Domain '{domain_name}' not found in indexes")

    if domain_name not in all_splits:
        raise ValueError(f"Domain '{domain_name}' not found in all_splits")

    query_embed = embed_model.encode([query], normalize_embeddings=True)
    query_embed = np.array(query_embed, dtype="float32")

    scores, ids = indexes[domain_name].search(query_embed, top_k)

    return [all_splits[domain_name][i].page_content for i in ids[0] if i != -1]
    '''

# Stage 4 – Reranking, BM25, Hyde, Repacking and Summarization

In [ ]:
# ── Sentence helpers ───────────────────────────────────────────
# def split_into_sentences(text):
#     text = str(text).strip()

#     if not text:
#         return []

#     # Protect common abbreviations
#     text = re.sub(r"\b(Dr|Mr|Mrs|Ms|Prof|Inc|Ltd|U\.S|Fig|Sec|Art|No)\.", r"\1<prd>", text)

#     sentences = [
#         s.replace("<prd>", ".").strip()
#         for s in re.split(r'(?<=[.!?])\s+', text)
#         if s.strip()
#     ]

#     return sentences


from nltk.tokenize import sent_tokenize

def split_into_sentences(text):
    return [
        s.strip()
        for s in sent_tokenize(str(text))
        if s.strip()
    ]

def count_total_sentences(text):
    return len(split_into_sentences(text))

In [ ]:
#@title Reranker
# ───────────────────────────────────────────────────────────────

#@title Reranker
# ───────────────────────────────────────────────────────────────

class MonoT5Reranker:
    def __init__(self, model_name=MONOT5_MODEL):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()

        # Token ids for true/false.
        # T5 tokenizers often use ▁true / ▁false, but fallback is added.
        self.true_id = self.tokenizer.convert_tokens_to_ids("▁true")
        self.false_id = self.tokenizer.convert_tokens_to_ids("▁false")

        # FIX: use real Python <, not HTML &lt;
        if self.true_id is None or self.true_id < 0:
            self.true_id = self.tokenizer.encode(
                "true",
                add_special_tokens=False
            )[0]

        if self.false_id is None or self.false_id < 0:
            self.false_id = self.tokenizer.encode(
                "false",
                add_special_tokens=False
            )[0]

        print(f"MonoT5 loaded: {model_name} on {self.device}")

    def score(self, query, document):
        """
        Score one query-document pair using MonoT5 true/false probability.
        """
        text = f"Query: {query} Document: {document} Relevant:"

        enc = self.tokenizer(
            text,
            return_tensors="pt",
            max_length=512,
            truncation=True
        ).to(self.device)

        with torch.no_grad():
            out = self.model.generate(
                **enc,
                max_new_tokens=1,
                return_dict_in_generate=True,
                output_scores=True
            )

        logits = out.scores[0][0]

        pair_logits = torch.stack([
            logits[self.false_id],
            logits[self.true_id]
        ])

        probs = torch.softmax(pair_logits, dim=0)

        return float(probs[1].item())

    def compute_scores(self, query, texts):
        """
        Score multiple documents.
        """
        return np.array(
            [self.score(query, text) for text in texts],
            dtype=float
        )

    def rerank(self, query, docs, top_k=3):
        """
        Rerank list of dict docs.
        """
        if not docs:
            return []

        texts = [
            d.get("text", "") if isinstance(d, dict) else str(d)
            for d in docs
        ]

        scores = self.compute_scores(query, texts)
        scores = np.asarray(scores, dtype=float).reshape(-1)

        if len(scores) != len(docs):
            raise ValueError(
                f"MonoT5 score count mismatch: "
                f"{len(scores)} scores for {len(docs)} docs"
            )

        top_k = min(top_k, len(docs))

        ranked_idx = np.argsort(scores)[::-1][:top_k]

        reranked = []

        for i in ranked_idx:
            item = docs[i]

            if isinstance(item, dict):
                new_item = dict(item)

                # Preserve original retrieval/hybrid score
                new_item["base_score"] = new_item.get("score", None)

                # Use reranker score as final score
                new_item["score"] = float(scores[i])
                new_item["rerank_score"] = float(scores[i])
                new_item["reranker_type"] = "monot5"

                reranked.append(new_item)

            else:
                reranked.append({
                    "text": str(item),
                    "base_score": None,
                    "score": float(scores[i]),
                    "rerank_score": float(scores[i]),
                    "reranker_type": "monot5"
                })

        return reranked


# Lazy-loaded global model
monot5_reranker = None

def get_monot5_reranker():
    global monot5_reranker

    if monot5_reranker is None:
        monot5_reranker = MonoT5Reranker(MONOT5_MODEL)

    return monot5_reranker


# ───────────────────────────────────────────────────────────────
# 2) TILDE-style Reranker
# ───────────────────────────────────────────────────────────────

tilde_reranker = None

def get_tilde_reranker():
    """
    Lazy-load a fast TILDE-style reranker.

    Note:
    This uses BAAI/bge-reranker-base as a practical plug-and-play
    alternative for TILDE-style reranking. True TILDE requires a
    separate indexing/scoring setup and is not as simple as HF CrossEncoder.
    """
    global tilde_reranker

    if tilde_reranker is None:
        from sentence_transformers import CrossEncoder

        device = "cuda" if torch.cuda.is_available() else "cpu"

        print(f"Loading TILDE-style reranker: {TILDE_MODEL} on {device}")

        tilde_reranker = CrossEncoder(
            TILDE_MODEL,
            device=device
        )

    return tilde_reranker


def tilde_compute_scores(query, texts):
    """
    Compute scores using TILDE-style CrossEncoder.
    """
    model = get_tilde_reranker()

    pairs = [
        (query, str(text))
        for text in texts
    ]

    scores = model.predict(
        pairs,
        show_progress_bar=False
    )

    return np.asarray(scores, dtype=float).reshape(-1)


# ───────────────────────────────────────────────────────────────
# 3) Unified reranking function
# ───────────────────────────────────────────────────────────────

def rerank_documents(query, documents, top_k=3):
    """
    Unified reranking interface.

    Supports:
    - RERANKER_TYPE = "monot5"
    - RERANKER_TYPE = "tilde"

    Input:
        documents: list[dict] or list[str]

    Output:
        list[dict] with:
        - text
        - score
        - rerank_score
        - reranker_type
        - base_score
    """

    if not documents:
        return []

    texts = [
        d.get("text", "") if isinstance(d, dict) else str(d)
        for d in documents
    ]

    reranker_type = RERANKER_TYPE.lower().strip()

    if reranker_type == "monot5":
        model = get_monot5_reranker()
        scores = model.compute_scores(query, texts)

    elif reranker_type == "tilde":
        scores = tilde_compute_scores(query, texts)

    else:
        raise ValueError(
            f"Unknown RERANKER_TYPE={RERANKER_TYPE}. "
            "Use 'monot5' or 'tilde'."
        )

    scores = np.asarray(scores, dtype=float).reshape(-1)

    if len(scores) != len(documents):
        raise ValueError(
            f"Reranker score count mismatch: "
            f"{len(scores)} scores for {len(documents)} documents"
        )

    top_k = min(top_k, len(documents))

    ranked_idx = np.argsort(scores)[::-1][:top_k]

    reranked = []

    for i in ranked_idx:
        item = documents[i]

        if isinstance(item, dict):
            new_item = dict(item)

            # Preserve original retrieval/hybrid score
            new_item["base_score"] = new_item.get("score", None)

            # Use reranker score as final score
            new_item["score"] = float(scores[i])
            new_item["rerank_score"] = float(scores[i])
            new_item["reranker_type"] = reranker_type

            reranked.append(new_item)

        else:
            reranked.append({
                "text": str(item),
                "base_score": None,
                "score": float(scores[i]),
                "rerank_score": float(scores[i]),
                "reranker_type": reranker_type,
            })

    return reranked


print("Unified reranker block loaded ✅")


# class MonoT5Reranker:
#     def __init__(self, model_name=MONOT5_MODEL):
#         self.tokenizer = AutoTokenizer.from_pretrained(model_name)
#         self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

#         self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#         self.model.to(self.device)
#         self.model.eval()

#         # Token ids for true/false.
#         # T5 tokenizers often use ▁true / ▁false, but fallback is added.
#         self.true_id = self.tokenizer.convert_tokens_to_ids("▁true")
#         self.false_id = self.tokenizer.convert_tokens_to_ids("▁false")

#         if self.true_id is None or self.true_id < 0:
#             self.true_id = self.tokenizer.encode("true", add_special_tokens=False)[0]

#         if self.false_id is None or self.false_id < 0:
#             self.false_id = self.tokenizer.encode("false", add_special_tokens=False)[0]

#         print(f"MonoT5 loaded: {model_name} on {self.device}")

#     def score(self, query, document):
#         """
#         Score one query-document pair using MonoT5 true/false probability.
#         """
#         text = f"Query: {query} Document: {document} Relevant:"

#         enc = self.tokenizer(
#             text,
#             return_tensors="pt",
#             max_length=512,
#             truncation=True
#         ).to(self.device)

#         with torch.no_grad():
#             out = self.model.generate(
#                 **enc,
#                 max_new_tokens=1,
#                 return_dict_in_generate=True,
#                 output_scores=True
#             )

#         logits = out.scores[0][0]

#         pair_logits = torch.stack([
#             logits[self.false_id],
#             logits[self.true_id]
#         ])

#         probs = torch.softmax(pair_logits, dim=0)

#         return float(probs[1].item())

#     def compute_scores(self, query, texts):
#         """
#         Score multiple documents.
#         """
#         return np.array(
#             [self.score(query, text) for text in texts],
#             dtype=float
#         )

#     def rerank(self, query, docs, top_k=3):
#         """
#         Rerank list of dict docs.
#         """
#         if not docs:
#             return []

#         texts = [
#             d["text"] if isinstance(d, dict) else d
#             for d in docs
#         ]

#         scores = self.compute_scores(query, texts)

#         ranked_idx = np.argsort(scores)[::-1][:top_k]

#         reranked = []
#         for i in ranked_idx:
#             item = docs[i]

#             if isinstance(item, dict):
#                 new_item = dict(item)
#                 new_item["rerank_score"] = float(scores[i])
#                 new_item["reranker_type"] = "monot5"
#                 reranked.append(new_item)
#             else:
#                 reranked.append({
#                     "text": item,
#                     "rerank_score": float(scores[i]),
#                     "reranker_type": "monot5"
#                 })

#         return reranked


# # Lazy-loaded global model
# monot5_reranker = None

# def get_monot5_reranker():
#     global monot5_reranker

#     if monot5_reranker is None:
#         monot5_reranker = MonoT5Reranker(MONOT5_MODEL)

#     return monot5_reranker


# # ───────────────────────────────────────────────────────────────
# # 2) TILDE-style Reranker
# # ───────────────────────────────────────────────────────────────

# tilde_reranker = None

# def get_tilde_reranker():
#     """
#     Lazy-load a fast TILDE-style reranker.

#     Note:
#     This uses BAAI/bge-reranker-base as a practical plug-and-play
#     alternative for TILDE-style reranking. True TILDE requires a
#     separate indexing/scoring setup and is not as simple as HF CrossEncoder.
#     """
#     global tilde_reranker

#     if tilde_reranker is None:
#         from sentence_transformers import CrossEncoder

#         device = "cuda" if torch.cuda.is_available() else "cpu"

#         print(f"Loading TILDE-style reranker: {TILDE_MODEL} on {device}")

#         tilde_reranker = CrossEncoder(
#             TILDE_MODEL,
#             device=device
#         )

#     return tilde_reranker


# def tilde_compute_scores(query, texts):
#     """
#     Compute scores using TILDE-style CrossEncoder.
#     """
#     model = get_tilde_reranker()

#     pairs = [(query, text) for text in texts]

#     scores = model.predict(
#         pairs,
#         show_progress_bar=False
#     )

#     return np.asarray(scores, dtype=float).reshape(-1)


# # ───────────────────────────────────────────────────────────────
# # 3) Unified reranking function
# # ───────────────────────────────────────────────────────────────

# def rerank_documents(query, documents, top_k=3):
#     """
#     Unified reranking interface.

#     Supports:
#     - RERANKER_TYPE = "monot5"
#     - RERANKER_TYPE = "tilde"

#     Input:
#         documents: list[dict] or list[str]

#     Output:
#         list[dict] with:
#         - text
#         - score
#         - rerank_score
#         - reranker_type
#         - base_score
#     """

#     if not documents:
#         return []

#     texts = [
#         d["text"] if isinstance(d, dict) else d
#         for d in documents
#     ]

#     reranker_type = RERANKER_TYPE.lower().strip()

#     if reranker_type == "monot5":
#         model = get_monot5_reranker()
#         scores = model.compute_scores(query, texts)

#     elif reranker_type == "tilde":
#         scores = tilde_compute_scores(query, texts)

#     else:
#         raise ValueError(
#             f"Unknown RERANKER_TYPE={RERANKER_TYPE}. "
#             "Use 'monot5' or 'tilde'."
#         )

#     scores = np.asarray(scores, dtype=float).reshape(-1)

#     if len(scores) != len(documents):
#         raise ValueError(
#             f"Reranker score count mismatch: "
#             f"{len(scores)} scores for {len(documents)} documents"
#         )

#     ranked_idx = np.argsort(scores)[::-1][:top_k]

#     reranked = []

#     for i in ranked_idx:
#         item = documents[i]

#         if isinstance(item, dict):
#             new_item = dict(item)

#             # Preserve original retrieval/hybrid score
#             new_item["base_score"] = new_item.get("score", None)

#             # Use reranker score as final score
#             new_item["score"] = float(scores[i])
#             new_item["rerank_score"] = float(scores[i])
#             new_item["reranker_type"] = reranker_type

#             reranked.append(new_item)

#         else:
#             reranked.append({
#                 "text": item,
#                 "base_score": None,
#                 "score": float(scores[i]),
#                 "rerank_score": float(scores[i]),
#                 "reranker_type": reranker_type,
#             })

#     return reranked



# print("Unified reranker block loaded ✅")


In [ ]:
# ───────────────────────────────────────────────────────────────
# Reciprocal Rank Fusion (RRF)
# ───────────────────────────────────────────────────────────────

def reciprocal_rank_fusion(
    dense_results,
    bm25_results,
    top_k=20,
    rrf_k=60
):
    """
    Fuse dense + BM25 rankings using RRF.

    Parameters
    ----------
    dense_results : dict
        {text: dense_score}

    bm25_results : dict
        {text: bm25_score}

    rrf_k : int
        RRF constant. Typical values: 60-100

    Returns
    -------
    list[dict]
    """

    rrf_scores = {}

    # Dense ranking
    dense_ranked = sorted(
        dense_results.items(),
        key=lambda x: x[1],
        reverse=True
    )

    for rank, (text, score) in enumerate(dense_ranked, start=1):
        rrf_scores.setdefault(
            text,
            {
                "text": text,
                "dense_score": score,
                "bm25_score": 0.0,
                "score": 0.0
            }
        )

        rrf_scores[text]["score"] += 1.0 / (rrf_k + rank)

    # BM25 ranking
    bm25_ranked = sorted(
        bm25_results.items(),
        key=lambda x: x[1],
        reverse=True
    )

    for rank, (text, score) in enumerate(bm25_ranked, start=1):
        rrf_scores.setdefault(
            text,
            {
                "text": text,
                "dense_score": 0.0,
                "bm25_score": score,
                "score": 0.0
            }
        )

        rrf_scores[text]["bm25_score"] = score
        rrf_scores[text]["score"] += 1.0 / (rrf_k + rank)

    fused = list(rrf_scores.values())

    fused.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return fused[:top_k]

In [ ]:
if RERANKER_TYPE == "monot5":
  # MonoT5
  monot5_reranker = MonoT5Reranker(
      MONOT5_MODEL)
else:
  # TILDE-style reranker
  from sentence_transformers import CrossEncoder

  tilde_reranker = CrossEncoder(
      TILDE_MODEL,
      device=device)

print("✅ All models loaded")

In [ ]:
# @title Hybrid, BM25, HyDE

import re
import time
import random
import numpy as np
from rank_bm25 import BM25Okapi


# ───────────────────────────────────────────────────────────────
# 1) BM25 Tokenizer
# ───────────────────────────────────────────────────────────────

def _tokenize(text):
    """
    Simple BM25 tokenizer.
    Lowercases text and extracts alphanumeric tokens.
    """
    return re.findall(r"\w+", str(text).lower())


# domain_name -> {"bm25": BM25Okapi, "texts": [str]}
bm25_indexes = {}


# ───────────────────────────────────────────────────────────────
# 2) Build BM25 Index Per Domain
# ───────────────────────────────────────────────────────────────

def build_bm25_index(domain_name, milvus_clients):
    """
    Build BM25 index for one domain by reading text chunks from Milvus.
    """

    if domain_name not in milvus_clients:
        print(f"  ❌ {domain_name} not found in milvus_clients")
        return

    client = milvus_clients[domain_name]
    collection_name = domain_name.lower()

    print(f"\n[{domain_name}] Building BM25 index...")
    print(f"  Collection: {collection_name}")

    # Ensure collection is loaded
    try:
        load_state = client.get_load_state(collection_name)
        print(f"  Load state: {load_state}")

        if load_state != "Loaded":
            client.load_collection(collection_name)
            print("  Collection loaded")

    except Exception as e:
        print(f"  Load-state check skipped: {e}")
        try:
            client.load_collection(collection_name)
            print("  Collection loaded")
        except Exception as load_err:
            print(f"  ⚠️ Collection load failed or not required: {load_err}")

    # Get row count
    try:
        stats = client.get_collection_stats(collection_name)
        n_rows = int(stats.get("row_count", 0))
    except Exception as e:
        print(f"  ❌ Could not read collection stats: {e}")
        return

    print(f"  Row count: {n_rows}")

    if n_rows == 0:
        print("  ❌ Skipped: empty collection")
        return

    # Query all texts
    try:
        results = client.query(
            collection_name=collection_name,
            filter="",
            limit=n_rows,
            output_fields=["text"]
        )
    except Exception as e:
        print(f"  ❌ Query failed while building BM25: {e}")
        return

    print(f"  Query returned: {len(results)} rows")

    if not results:
        print("  ❌ Skipped: query returned no rows")
        return

    texts = [
        r.get("text", "")
        for r in results
        if r.get("text", "")
    ]

    if not texts:
        print("  ❌ Skipped: no valid text fields")
        return

    tokenized = [
        _tokenize(t)
        for t in texts
    ]

    bm25_indexes[domain_name] = {
        "bm25": BM25Okapi(tokenized),
        "texts": texts,
    }

    print(f"  ✅ BM25 index built: {len(texts)} docs")


def build_all_bm25_indexes(milvus_clients):
    """
    Build BM25 indexes for all loaded Milvus domains.
    """

    print(f"\nProcessing {len(milvus_clients)} domains for BM25...")

    bm25_indexes.clear()

    for domain_name in milvus_clients.keys():
        build_bm25_index(
            domain_name=domain_name,
            milvus_clients=milvus_clients
        )

    print("\n✅ Done. BM25 indexes built for:")
    print(list(bm25_indexes.keys()))


# Run manually after milvus_clients are loaded:
# build_all_bm25_indexes(milvus_clients)

# ───────────────────────────────────────────────────────────────
# 3) HyDE: Generate Hypothetical Answer
# ───────────────────────────────────────────────────────────────

def generate_hyde(query, llm_client, model_name=None, max_retries=3):
    """
    Generate a short hypothetical passage for HyDE retrieval.

    Always returns a string.
    Returns "" on failure so retrieve() can safely fall back to original query.
    """

    model = model_name or globals().get("MODEL_NAME")

    if not model:
        print("HyDE skipped: MODEL_NAME not found.")
        return ""

    prompt = f"""
Write a brief, factual passage that would answer this question.
Use specific terms that may help retrieve relevant documents.
Do not say "I don't know".
Do not invent unsupported facts.
Keep it under 4 sentences.

Question:
{query}

Passage:
""".strip()

    last_error = None

    for attempt in range(max_retries):
        try:
            response = llm_client.chat.completions.create(
                model=model,
                messages=[
                    {
                        "role": "system",
                        "content": "You write short hypothetical answer passages for retrieval."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    },
                ],
                temperature=0.2,
                max_tokens=300,
            )

            content = response.choices[0].message.content

            if content is None:
                return ""

            content = str(content).strip()

            if not content:
                return ""

            return content

        except Exception as e:
            last_error = e
            msg = str(e)

            if "tokens per day" in msg.lower():
                raise RuntimeError(f"TPD_EXHAUSTED: {msg}")

            if (
                "429" in msg
                or "rate_limit_exceeded" in msg
                or "503" in msg
                or "over capacity" in msg.lower()
                or "internal_server_error" in msg
                or "502" in msg
                or "504" in msg
                or "gateway" in msg.lower()
            ):
                wait_sec = (2 ** attempt) + random.uniform(0.2, 0.8)
                print(
                    f"HyDE retrying in {wait_sec:.1f}s due to API limit/capacity..."
                )
                time.sleep(wait_sec)
                continue

            print(f"HyDE generation failed: {e}")
            return ""

    print(f"HyDE failed after {max_retries} retries: {last_error}")
    return ""


# ───────────────────────────────────────────────────────────────
# 4) Score Normalization
# ───────────────────────────────────────────────────────────────

def _normalize(scores):
    """
    Normalize scores to [0, 1].
    If all scores are equal, returns zeros.
    """

    arr = np.array(scores, dtype=float)

    if len(arr) == 0:
        return arr

    if arr.max() == arr.min():
        return np.zeros_like(arr)

    return (arr - arr.min()) / (arr.max() - arr.min())


# ───────────────────────────────────────────────────────────────
# 5) Hybrid Retrieval: Dense + BM25
# ───────────────────────────────────────────────────────────────

def hybrid_search(query, domain_name, embed_model, top_k=20, alpha=0.5):
    """
    Combine dense Milvus retrieval with BM25 lexical retrieval.

    Parameters
    ----------
    query : str
        Search query.
    domain_name : str
        Domain name.
    embed_model : SentenceTransformer
        Embedding model.
    top_k : int
        Number of results to return.
    alpha : float
        Dense score weight.
        BM25 weight is 1 - alpha.

    Returns
    -------
    list[dict]
        Each result contains:
        - text
        - score
        - dense_score
        - bm25_score
        - alpha
    """

    if domain_name not in milvus_clients:
        raise ValueError(f"Domain '{domain_name}' not loaded in milvus_clients.")

    client = milvus_clients[domain_name]
    collection_name = domain_name.lower()

    # Ensure collection is loaded
    try:
        if client.get_load_state(collection_name) != "Loaded":
            client.load_collection(collection_name)
    except Exception:
        try:
            client.load_collection(collection_name)
        except Exception:
            pass

    # ── Dense search ───────────────────────────────────────────
    query_embed = embed_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    dense_hits = client.search(
        collection_name=collection_name,
        data=query_embed.tolist(),
        limit=top_k,
        output_fields=["text"],
        search_params={
            "metric_type": "IP",
            "params": {}
        }
    )

    dense_results = {}

    for hit in dense_hits[0]:
        text = hit.entity.get("text", "")

        if text:
            dense_results[text] = float(hit.distance)

    # ── BM25 search ────────────────────────────────────────────
    bm25_obj = bm25_indexes.get(domain_name)

    # Fallback to dense-only if BM25 is unavailable
    if bm25_obj is None:
        items = sorted(
            dense_results.items(),
            key=lambda x: x[1],
            reverse=True
        )

        return [
            {
                "text": text,
                "score": float(score),
                "dense_score": float(score),
                "bm25_score": 0.0,
                "alpha": alpha,
                "hybrid_fallback": True,
            }
            for text, score in items[:top_k]
        ]

    bm25 = bm25_obj["bm25"]
    bm25_texts = bm25_obj["texts"]

    bm25_scores = bm25.get_scores(
        _tokenize(query)
    )

    top_bm25_idx = np.argsort(
        bm25_scores
    )[::-1][:top_k]

    bm25_results = {
        bm25_texts[i]: float(bm25_scores[i])
        for i in top_bm25_idx
    }
    # ── Fusion ─────────────────────────────────────────────────

    if ENABLE_RRF:

        return reciprocal_rank_fusion(
            dense_results=dense_results,
            bm25_results=bm25_results,
            top_k=top_k,
            rrf_k=RRF_K
        )

    else:
        # Original min-max fusion

        all_texts = sorted(
            set(dense_results.keys()) | set(bm25_results.keys())
        )

        d_vals = [
            dense_results.get(t, 0.0)
            for t in all_texts
        ]

        b_vals = [
            bm25_results.get(t, 0.0)
            for t in all_texts
        ]

        d_norm = _normalize(d_vals)
        b_norm = _normalize(b_vals)

        combined = []

        for i, text in enumerate(all_texts):
            combined_score = (
                alpha * d_norm[i]
                + (1.0 - alpha) * b_norm[i]
            )

            combined.append({
                "text": text,
                "score": float(combined_score),
                "dense_score": float(d_vals[i]),
                "bm25_score": float(b_vals[i]),
                "alpha": alpha,
            })

        combined.sort(
            key=lambda x: x["score"],
            reverse=True
        )

    return combined[:top_k]



print("Hybrid / BM25 / HyDE block loaded ✅")


In [ ]:
build_all_bm25_indexes(milvus_clients)

In [ ]:

# @title Repacking
# ── Repacking function ─────────────────────────────────────────
def repack_documents(docs, strategy="sides"):
    """
    Reorder retrieved documents before LLM prompt construction.

    Strategies:
    - forward : most relevant first (no change)
    - reverse : most relevant last (near the question)
    - sides   : U-shape — high relevance at edges, low in middle

    Parameters
    ----------
    docs : list[dict] or list[str]
        Documents sorted by relevance (most relevant first)
    strategy : str
        "forward" | "reverse" | "sides"

    Returns
    -------
    list[dict] or list[str]
        Reordered documents
    """
    if not docs:
        return []

    if strategy == "forward":
        return docs

    if strategy == "reverse":
        return docs[::-1]

    if strategy == "sides":
        # U-shape: alternate between left and right
        # Most relevant docs go to edges, least relevant to middle
        n = len(docs)
        result = [None] * n
        left, right = 0, n - 1

        for i, doc in enumerate(docs):
            if i % 2 == 0:
                result[left] = doc
                left += 1
            else:
                result[right] = doc
                right -= 1

        return result

    raise ValueError(f"Unknown REPACK_STRATEGY: {strategy}")

In [ ]:
# @title Summarization

import re
import torch
import numpy as np

# ─────────────────────────────────────────────────────────────────
# Feature flags / configs
# ─────────────────────────────────────────────────────────────────

ENABLE_SUMMARIZATION = False
SUMMARIZATION_TYPE   = "recomp"   # "recomp" | "longllmlingua"

# RECOMP config
RECOMP_TOP_K_SENTS      = 6
RECOMP_MIN_SCORE        = 0.0
RECOMP_GROUNDING_BOOST  = 0.15
RECOMP_MIN_KEEP_RATIO   = 0.50
RECOMP_KEEP_CRITICAL    = True

# LongLLMLingua config
LLMLINGUA_RATE = 0.5   # keep 50% of tokens


# ─────────────────────────────────────────────────────────────────
# 1) Grounding signal helpers
# ─────────────────────────────────────────────────────────────────

GROUNDING_PATTERNS = [
    # Important qualifiers / factual constraints
    r"\b(?:must|should|shall|cannot|can't|never|always|only|except|unless|required|recommended)\b",

    # Warnings / cautions
    r"\b(?:warning|caution|note|important|attention)\b",

    # Negations
    r"\b(?:do not|don't|does not|did not|not allowed|not recommended|never)\b",

    # Time / units / measurements
    r"\b\d+(?:\.\d+)?\s*(?:%|percent|seconds?|minutes?|hours?|days?|weeks?|months?|years?)\b",
    r"\b\d+(?:\.\d+)?\s*(?:GB|MB|KB|TB|kg|g|mg|mm|cm|m|km|degrees?|°C|°F)\b",

    # Currency / financial values
    r"[$€£¥]\s*\d+(?:,\d{3})*(?:\.\d+)?",
    r"\b\d+(?:,\d{3})*(?:\.\d+)?\s*(?:dollars?|rupees?|crores?|lakhs?|million|billion)\b",

    # Years and large numbers
    r"\b(?:19|20)\d{2}\b",
    r"\b\d{2,}\b",

    # Product / model / code-like patterns
    r"\b[A-Z]{2,}[-_]?\d+[A-Z0-9-]*\b",
    r"\b[A-Z0-9]{3,}[-_][A-Z0-9]{2,}\b",

    # Acronyms
    r"\b[A-Z]{3,}\b",
]

GROUNDING_REGEX = re.compile("|".join(GROUNDING_PATTERNS), re.IGNORECASE)


def _count_grounding_signals(sentence):
    """
    Count numbers, dates, model names, acronyms, warnings, qualifiers, etc.
    """
    return len(GROUNDING_REGEX.findall(str(sentence)))


def _is_critical_sentence(sentence):
    """
    Preserve warning/caution/negative/mandatory sentences even if semantic score is low.
    """
    critical_pattern = re.compile(
        r"\b(?:warning|caution|important|must|must not|cannot|can't|do not|don't|never|only|except|unless|required)\b",
        re.IGNORECASE
    )
    return bool(critical_pattern.search(str(sentence)))


# ─────────────────────────────────────────────────────────────────
# 2) Grounded RECOMP-style Extractive Summarization
# ─────────────────────────────────────────────────────────────────

def recomp_summarize(
    query,
    docs,
    embed_model,
    top_k=6,
    min_score=0.0,
    grounding_boost=0.15,
    min_keep_ratio=0.50,
    keep_critical=True
):
    """
    Grounded RECOMP-style extractive summarization.

    Selects query-relevant sentences while preserving factual grounding details:
    - numbers
    - dates
    - percentages
    - financial values
    - product/model names
    - warnings/cautions
    - negations
    - qualifiers

    Returns
    -------
    str
        Compressed context string.
    """

    if docs and isinstance(docs[0], dict):
        texts = [d.get("text", "") for d in docs]
    else:
        texts = docs

    sentences = []
    for doc in texts:
        sentences.extend(split_into_sentences(doc))

    sentences = [s.strip() for s in sentences if s.strip()]

    if not sentences:
        return ""

    if embed_model is None:
        raise ValueError("embed_model is required for RECOMP summarization.")

    # Query-sentence semantic similarity
    query_emb = embed_model.encode(
        [query],
        normalize_embeddings=True
    )

    sent_emb = embed_model.encode(
        sentences,
        normalize_embeddings=True
    )

    base_scores = (query_emb @ sent_emb.T).flatten()

    # Boost grounding-heavy sentences
    grounding_scores = np.array([
        _count_grounding_signals(s) * grounding_boost
        for s in sentences
    ])

    final_scores = base_scores + grounding_scores

    # Critical sentences to always preserve
    critical_indices = set()
    if keep_critical:
        critical_indices = {
            i for i, s in enumerate(sentences)
            if _is_critical_sentence(s)
        }

    # Filter by min score
    valid_idx = np.where(final_scores >= min_score)[0]

    if len(valid_idx) == 0:
        valid_idx = np.array([int(np.argmax(final_scores))])

    # Keep both top_k and minimum keep ratio
    min_keep_count = max(
        top_k,
        int(np.ceil(len(sentences) * min_keep_ratio))
    )

    actual_keep = min(min_keep_count, len(sentences))

    valid_scores = final_scores[valid_idx]
    ranked_valid = np.argsort(valid_scores)[::-1]

    selected_idx = list(valid_idx[ranked_valid][:actual_keep])

    # Force include critical grounding sentences
    selected_idx.extend(list(critical_indices))

    # Deduplicate and preserve original document order
    selected_idx = sorted(set(selected_idx))

    summary = " ".join([sentences[i] for i in selected_idx])
    return summary


# ─────────────────────────────────────────────────────────────────
# 3) LongLLMLingua-style Abstractive Compression
# ─────────────────────────────────────────────────────────────────

llmlingua_compressor = None


def _get_llmlingua():
    """
    Lazily load LLMLingua compressor.
    Auto-detects CPU/GPU.
    """
    global llmlingua_compressor

    if llmlingua_compressor is None:
        try:
            from llmlingua import PromptCompressor

            device = "cuda" if torch.cuda.is_available() else "cpu"

            llmlingua_compressor = PromptCompressor(
                model_name="microsoft/llmlingua-2-bert-base-multilingual-cased-meetingbank",
                use_llmlingua2=True,
                device_map=device
            )

            print(f"LongLLMLingua loaded on {device}")

        except ImportError:
            raise ImportError("Install with: !pip install llmlingua")

    return llmlingua_compressor


def llmlingua_compress(query, docs, rate=0.5):
    """
    Compress documents using LLMLingua-2.

    Compresses each doc independently to avoid 512-token overflow.
    """

    if docs and isinstance(docs[0], dict):
        texts = [d.get("text", "") for d in docs]
    else:
        texts = docs

    compressor = _get_llmlingua()

    compressed_parts = []

    for text in texts:
        if not text or not text.strip():
            continue

        try:
            result = compressor.compress_prompt(
                text,
                question=query,
                rate=rate
            )
            compressed_parts.append(result["compressed_prompt"])

        except Exception as e:
            print(f"  LLMLingua failed on one chunk, keeping original. Error: {e}")
            compressed_parts.append(text)

    return "\n\n".join(compressed_parts)


# ─────────────────────────────────────────────────────────────────
# 4) Unified summarization wrapper
# ─────────────────────────────────────────────────────────────────

def summarize_docs(query, docs, embed_model=None, llm_client=None):
    """
    Apply summarization based on feature flag.

    Returns
    -------
    list[str]
        Compressed/summarized documents.
        Always returns a list for downstream compatibility.
    """

    if not ENABLE_SUMMARIZATION:
        return [
            d.get("text", "") if isinstance(d, dict) else d
            for d in docs
        ]

    if SUMMARIZATION_TYPE == "recomp":
        embed_model = embed_model or globals().get("embed_model")

        summary = recomp_summarize(
            query=query,
            docs=docs,
            embed_model=embed_model,
            top_k=RECOMP_TOP_K_SENTS,
            min_score=RECOMP_MIN_SCORE,
            grounding_boost=RECOMP_GROUNDING_BOOST,
            min_keep_ratio=RECOMP_MIN_KEEP_RATIO,
            keep_critical=RECOMP_KEEP_CRITICAL
        )

        return [summary] if summary else []

    elif SUMMARIZATION_TYPE == "longllmlingua":
        compressed = llmlingua_compress(
            query=query,
            docs=docs,
            rate=LLMLINGUA_RATE
        )

        return [compressed] if compressed else []

    else:
        raise ValueError(f"Unknown SUMMARIZATION_TYPE: {SUMMARIZATION_TYPE}")


print("Summarization functions defined ✅")

In [ ]:
# @title Query Classification, Rewriting and Decomposition

# ───────────────────────────────────────────────────────────────
# Query Classification / Rewriting / Decomposition
# ───────────────────────────────────────────────────────────────

import json
import re

# Optional separate models
QUERY_REWRITE_MODEL = globals().get("MODEL_NAME", None)
QUERY_DECOMPOSE_MODEL = globals().get("MODEL_NAME", None)
DIRECT_LLM_MODEL = globals().get("MODEL_NAME", None)


def _safe_message_content(response):
    """
    Safely extract assistant content from OpenAI/OpenRouter/Groq-like response.
    Returns "" if content is missing.
    """
    try:
        msg = response.choices[0].message
        content = getattr(msg, "content", None)

        if content is None:
            return ""

        return str(content).strip()

    except Exception:
        return ""


def classify_query(query: str, domain_name: str = None) -> str:
    """
    Classifies query as:
    - "RAG": needs retrieval
    - "LLM": can be answered directly by LLM

    For known RAGBench project domains, default to RAG because Legal,
    Biomedical, Finance, and Support queries often look like general questions
    but still require dataset-grounded retrieval.
    """

    if not ENABLE_QUERY_CLASSIFICATION:
        return "RAG"

    query_l = str(query).lower().strip()

    rag_domains = {
        "Customer_Support",
        "Finance",
        "General_Knowledge",
        "Legal_Contracts",
        "Bio_Medical",
    }

    # For your benchmark/domain-specific evaluation, prefer RAG.
    if domain_name in rag_domains:
        return "RAG"

    llm_keywords = [
        "who is",
        "what is",
        "when was",
        "where is",
        "define",
        "explain",
        "tell me about",
        "what are",
        "why is",
        "history of",
        "meaning of",
        "list of",
        "difference between",
        "pros and cons",
        "best way to",
        "examples of",
        "how does",
        "what causes",
        "benefits of",
        "drawbacks of",
        "famous for",
        "biography of",
        "origin of",
        "summary of",
        "purpose of",
        "importance of",
        "types of",
        "facts about",
        "impact of",
    ]

    if any(keyword in query_l for keyword in llm_keywords):
        return "LLM"

    return "RAG"


def ask_llm(question, llm_client, model_name=None, max_tokens=500):
    """
    Direct LLM answer without retrieval.
    Used only when classifier routes query to LLM.
    """

    model = model_name or DIRECT_LLM_MODEL or globals().get("MODEL_NAME")

    if not model:
        raise ValueError("No model configured for ask_llm.")

    try:
        response = llm_client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a concise, factual assistant. "
                        "Answer clearly. If unsure, say so."
                    ),
                },
                {
                    "role": "user",
                    "content": question,
                },
            ],
            temperature=0.2,
            max_tokens=max_tokens,
        )

        content = _safe_message_content(response)

        if not content:
            return "I could not generate a reliable answer."

        return content

    except Exception as e:
        print(f"Direct LLM call failed: {e}")
        return "I could not generate a reliable answer."


def rewrite_query(query, domain_name, llm_client, model_name=None):
    """
    Rewrites a user question into a clearer standalone retrieval query.
    Returns original query if rewrite fails or if rewrite is unnecessary.
    """

    if not ENABLE_QUERY_REWRITING:
        return query

    model = model_name or QUERY_REWRITE_MODEL or globals().get("MODEL_NAME")

    if not model:
        return query

    prompt = f"""
Rewrite the question only when necessary to improve document retrieval.

Domain:
{domain_name}

Rules:
- Preserve the complete meaning and question form.
- Correct grammar and resolve ambiguity.
- Do not shorten an already clear question.
- Preserve names, product names, dates, numbers, legal terms, biomedical terms, financial terms, and technical terms.
- Do not answer the question.
- Do not add unsupported information.
- If the query is already clear, return it unchanged.
- Return only the rewritten query.

Original question:
{query}
""".strip()

    try:
        response = llm_client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": "You rewrite questions for semantic document retrieval.",
                },
                {
                    "role": "user",
                    "content": prompt,
                },
            ],
            temperature=0.0,
            max_tokens=120,
        )

        rewritten = _safe_message_content(response)

        if not rewritten:
            return query

        if len(rewritten) > 500:
            return query

        return rewritten.strip()

    except Exception as error:
        print(f"Query rewriting failed: {error}")
        return query


# def _parse_subqueries(raw_text, original_query):
#     """
#     Parse subqueries from JSON/list/plain text output.
#     Always returns a non-empty list.
#     """

#     if raw_text is None:
#         return [original_query]

#     text = str(raw_text).strip()

#     if not text:
#         return [original_query]

#     # Remove markdown fences
#     text = text.replace("```json", "").replace("```", "").strip()

#     # Try JSON first
#     try:
#         parsed = json.loads(text)

#         if isinstance(parsed, list):
#             subqueries = parsed

#         elif isinstance(parsed, dict):
#             subqueries = (
#                 parsed.get("subqueries")
#                 or parsed.get("queries")
#                 or parsed.get("questions")
#                 or []
#             )

#         else:
#             subqueries = []

#         subqueries = [
#             str(q).strip()
#             for q in subqueries
#             if str(q).strip()
#         ]

#         return subqueries or [original_query]

#     except Exception:
#         pass

#     # Fallback: parse numbered or bulleted lines
#     lines = [
#         re.sub(r"^\s*[-*\d.)]+\s*", "", line).strip()
#         for line in text.splitlines()
#         if line.strip()
#     ]

#     lines = [
#         line
#         for line in lines
#         if len(line) > 3
#     ]

#     return lines or [original_query]


# def decompose_query(query, llm_client, domain=None, model=None, max_subqueries=None):
#     """
#     Decompose complex query into simpler retrieval-focused subqueries.
#     Returns list[str].
#     """

#     if not ENABLE_QUERY_DECOMPOSITION:
#         return [query]

#     model = model or QUERY_DECOMPOSE_MODEL or globals().get("MODEL_NAME")
#     max_subqueries = max_subqueries or MAX_SUBQUERIES

#     if not model:
#         return [query]

#     prompt = f"""
# Decompose the following question into at most {max_subqueries} focused search queries for retrieval.

# Domain:
# {domain}

# Rules:
# - Only decompose if the question has multiple parts or requires multiple evidence points.
# - If it is already simple, return a list with the original question only.
# - Preserve dates, numbers, names, legal terms, biomedical terms, financial terms, and technical terms.
# - Do not answer the question.
# - Return only valid JSON as a list of strings.

# Question:
# {query}
# """.strip()

#     try:
#         response = llm_client.chat.completions.create(
#             model=model,
#             messages=[
#                 {
#                     "role": "system",
#                     "content": "You decompose complex questions into retrieval-focused subqueries.",
#                 },
#                 {
#                     "role": "user",
#                     "content": prompt,
#                 },
#             ],
#             temperature=0.0,
#             max_tokens=250,
#         )

#         raw = _safe_message_content(response)

#         subqueries = _parse_subqueries(
#             raw,
#             original_query=query
#         )

#         subqueries = subqueries[:max_subqueries]

#         return subqueries or [query]

#     except Exception as e:
#         print(f"Query decomposition failed: {e}")
#         return [query]


In [ ]:
# ───────────────────────────────────────────────────────────────
# Robust Query Decomposition Parser + Decompose Function
# ───────────────────────────────────────────────────────────────

import json
import re


def _clean_subquery_text(text):
    """
    Clean one candidate subquery string.
    """

    if text is None:
        return ""

    text = str(text).strip()

    # Remove markdown/code fences
    text = text.replace("```json", "").replace("```", "").strip()

    # Remove trailing commas
    text = text.rstrip(",")

    # Remove surrounding quotes
    text = text.strip('"').strip("'").strip()

    # Remove common bullet/number prefixes
    text = re.sub(r"^\s*[-*]\s*", "", text)
    text = re.sub(r"^\s*\d+[\).\:\-]\s*", "", text)

    return text.strip()


def _looks_like_explanation_line(text):
    """
    Filters out non-query explanation lines generated by LLMs.
    """

    if not text:
        return True

    text_l = text.lower().strip()

    bad_prefixes = [
        "here are",
        "here is",
        "decomposed",
        "search queries",
        "the decomposed",
        "queries:",
        "subqueries:",
        "output:",
        "json:",
        "answer:",
    ]

    if any(text_l.startswith(prefix) for prefix in bad_prefixes):
        return True

    # Ignore bare labels
    if text_l in {
        "queries",
        "subqueries",
        "search queries",
        "decomposed search queries",
    }:
        return True

    return False


def _parse_json_object_line(line):
    """
    Handles lines like:
        {"query": "Meloni"},
        {"question": "Who is Modi?"}
    """

    line = _clean_subquery_text(line)

    if not line:
        return None

    # Try direct JSON object
    try:
        obj = json.loads(line)

        if isinstance(obj, dict):
            for key in ["query", "question", "subquery", "search_query"]:
                if key in obj and str(obj[key]).strip():
                    return str(obj[key]).strip()

        if isinstance(obj, str):
            return obj.strip()

    except Exception:
        pass

    # Try to extract {"query": "..."} via regex if malformed
    m = re.search(
        r'"(?:query|question|subquery|search_query)"\s*:\s*"([^"]+)"',
        line
    )

    if m:
        return m.group(1).strip()

    return None


def _split_multi_question_locally(query, max_subqueries=None):
    """
    Handles simple multi-question input without spending LLM tokens.

    Example:
        "Who is Meloni? Who is Modi"
    Returns:
        ["Who is Meloni?", "Who is Modi?"]
    """

    max_subqueries = max_subqueries or MAX_SUBQUERIES

    query = str(query).strip()

    # Split on question marks while preserving question form
    parts = [
        p.strip()
        for p in re.split(r"\?\s*", query)
        if p.strip()
    ]

    if len(parts) <= 1:
        return None

    subqueries = []

    for p in parts[:max_subqueries]:

        # If original part already looks like question, restore '?'
        if not p.endswith("?"):
            p = p + "?"

        subqueries.append(p)

    return subqueries


def _parse_subqueries(raw_text, original_query, max_subqueries=None):
    """
    Robustly parse subqueries from:
    - valid JSON list
    - JSON object with 'subqueries'
    - JSON object lines like {"query": "..."}
    - numbered/bulleted text
    - messy LLM output with explanation lines

    Always returns a non-empty list.
    """

    max_subqueries = max_subqueries or MAX_SUBQUERIES

    if raw_text is None:
        return [original_query]

    text = str(raw_text).strip()

    if not text:
        return [original_query]

    # Remove markdown fences
    text = (
        text.replace("```json", "")
            .replace("```", "")
            .strip()
    )

    # ----------------------------------------------------------
    # 1. Try full JSON parsing first
    # ----------------------------------------------------------
    try:
        parsed = json.loads(text)

        if isinstance(parsed, list):
            subqueries = []

            for item in parsed:

                if isinstance(item, dict):
                    for key in ["query", "question", "subquery", "search_query"]:
                        if key in item and str(item[key]).strip():
                            subqueries.append(str(item[key]).strip())
                            break

                elif isinstance(item, str):
                    subqueries.append(item.strip())

            subqueries = [
                _clean_subquery_text(q)
                for q in subqueries
                if _clean_subquery_text(q)
            ]

            return subqueries[:max_subqueries] or [original_query]

        elif isinstance(parsed, dict):
            raw_list = (
                parsed.get("subqueries")
                or parsed.get("queries")
                or parsed.get("questions")
                or parsed.get("search_queries")
                or []
            )

            if isinstance(raw_list, list):
                subqueries = [
                    _clean_subquery_text(q)
                    for q in raw_list
                    if _clean_subquery_text(q)
                ]

                return subqueries[:max_subqueries] or [original_query]

    except Exception:
        pass

    # ----------------------------------------------------------
    # 2. Parse line-by-line
    # ----------------------------------------------------------
    subqueries = []

    for raw_line in text.splitlines():

        line = _clean_subquery_text(raw_line)

        if not line:
            continue

        if _looks_like_explanation_line(line):
            continue

        # Handle JSON object line
        obj_query = _parse_json_object_line(line)

        if obj_query:
            obj_query = _clean_subquery_text(obj_query)

            if obj_query and not _looks_like_explanation_line(obj_query):
                subqueries.append(obj_query)

            continue

        # Skip lines that are still JSON-like but not parseable
        if line.startswith("{") or line.endswith("}"):
            continue

        # Skip lines that contain only brackets
        if line in {"[", "]", "{", "}"}:
            continue

        # Normal text line
        subqueries.append(line)

    # Deduplicate while preserving order
    deduped = []

    for q in subqueries:
        q = _clean_subquery_text(q)

        if q and q not in deduped:
            deduped.append(q)

    if deduped:
        return deduped[:max_subqueries]

    return [original_query]


def decompose_query(
    query,
    llm_client,
    domain=None,
    model=None,
    max_subqueries=None
):
    """
    Decompose complex query into simpler retrieval-focused subqueries.

    Key fixes:
    - Handles "Who is X? Who is Y?" locally.
    - Forces JSON-list prompt.
    - Robustly parses messy LLM outputs.
    - Filters explanation lines like "Here are the decomposed search queries:".
    - Extracts query values from {"query": "..."} outputs.
    """

    if not ENABLE_QUERY_DECOMPOSITION:
        return [query]

    max_subqueries = max_subqueries or MAX_SUBQUERIES

    # ----------------------------------------------------------
    # 1. Local split for obvious multi-question cases
    # ----------------------------------------------------------
    local_split = _split_multi_question_locally(
        query,
        max_subqueries=max_subqueries
    )

    if local_split:
        return local_split

    model = model or QUERY_DECOMPOSE_MODEL or globals().get("MODEL_NAME")

    if not model:
        return [query]

    prompt = f"""
Decompose the question into at most {max_subqueries} retrieval-focused search queries.

Return ONLY a valid JSON list of strings.

Valid output example:
[
  "Who is Meloni?",
  "Who is Modi?"
]

Rules:
- Do not include explanations.
- Do not include introductory text.
- Do not include markdown.
- Do not return objects like {{"query": "..."}}
- Return only a JSON list of strings.
- If the question is already simple, return a JSON list with the original question only.
- Preserve dates, numbers, names, legal terms, biomedical terms, financial terms, and technical terms.
- Do not answer the question.

Domain:
{domain}

Question:
{query}
""".strip()

    try:
        response = llm_client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You decompose complex questions into retrieval-focused "
                        "search queries and return only JSON."
                    ),
                },
                {
                    "role": "user",
                    "content": prompt,
                },
            ],
            temperature=0.0,
            max_tokens=250,
        )

        raw = _safe_message_content(response)

        subqueries = _parse_subqueries(
            raw_text=raw,
            original_query=query,
            max_subqueries=max_subqueries
        )

        return subqueries[:max_subqueries] or [query]

    except Exception as e:
        print(f"Query decomposition failed: {e}")
        return [query]

# Stage 5 – Generator and Prompt Design

In [ ]:
# ── Prompt templates ────────────────────────────────────────────
def _build_prompt(context, question, strategy="short"):
    if strategy == "short":
        return f"""
Answer the question using the provided context.

Context:
{context}

Question:
{question}
""".strip()

    elif strategy == "long":
        return f"""
You are a chatbot providing answers to user queries. You will be given one or more context documents, and a question. \
Use the information in the documents to answer the question.
If the documents do not provide enough information for you to answer the question, then say \
"The documents are missing some of the information required to answer the question." Don't quote any external knowledge that is \
not in the documents. Don't try to make up an answer.

Context Documents:
{context}

Question: {question}
""".strip()

    elif strategy == "long_cot":
        return f"""
You are a chatbot providing answers to user queries. You will be given one or more context documents, and a question. \
Use the information in the documents to answer the question.
If the documents do not provide enough information for you to answer the question, then say \
"The documents are missing some of the information required to answer the question." Don't quote any external knowledge that is \
not in the documents. Don't try to make up an answer.

Think step by step and explain your reasoning, quoting the documents when necessary.

Context Documents:
{context}

Question: {question}
""".strip()

    else:
        raise ValueError(f"Unknown PROMPT_STRATEGY: {strategy}")


In [ ]:
# ── Updated ask_rag ─────────────────────────────────────────────
def ask_rag(context, question, llm_client, strategy=None):
    """
    RAG generation with configurable prompt strategy.

    Parameters
    ----------
    context : str
        Concatenated retrieved documents
    question : str
        User query
    llm_client : object
        Groq client
    strategy : str, optional
        Override global PROMPT_STRATEGY for this call.
        Options: "short" | "long" | "long_cot"
    """
    strategy = strategy or PROMPT_STRATEGY
    prompt = _build_prompt(context, question, strategy)

    response = llm_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a helpful RAG assistant"},
            {"role": "user", "content": prompt},
        ],
        temperature=0.3,
        max_tokens=400,
    )
    return response.choices[0].message.content

In [ ]:
# def ask_llm(question, llm_client):
#   response = llm_client.chat.completions.create(
#       model=MODEL_NAME,
#       messages=[
#           {"role": "system", "content": "You are a helpful LLM assistant"},
#           {"role": "user", "content": question},
#       ],
#       temperature=0.3,
#   )
#   return response.choices[0].message.content

In [ ]:
print("All Domains and Datasets")
print("=" * 50)

for domain_name, domain_datasets in ragbench_by_domain.items():
    print(f"\nDomain: {domain_name}")
    print(f"Number of datasets: {len(domain_datasets)}")
    print("Datasets:", ", ".join(domain_datasets.keys()))

In [ ]:
results = retrieve(
    "How do I enter Ambient Mode on my TV?",
    domain_name="Customer_Support",
    embed_model=embed_model,
    llm_client=llm_client,
    top_k=3
)

for i, r in enumerate(results, 1):
    rerank_score = r.get("rerank_score")
    score = r.get("score")
    print(f"\n--- Rank {i} ---")
    print(f"Rerank: {rerank_score:.4f}" if rerank_score else f"Score: {score:.4f}")
    pprint.pprint(f"Text: {r['text'][:200]}")

In [ ]:
# ENABLE_QUERY_CLASSIFICATION = True
# ENABLE_QUERY_REWRITING = True
# ENABLE_QUERY_DECOMPOSITION = True
# MAX_SUBQUERIES = 3

# result = answer_question_with_classifier(
#     question="What are the termination rights and notice requirements in the agreement?",
#     domain_name="Legal_Contracts",
#     llm_client=llm_client,
#     embed_model=embed_model,
#     top_k=3,
#     rag_strategy="short"
# )

# print(result["answer"])

In [ ]:
# rag_question_1 = "Who is Meloni? Who is Modi"
# result_rag_1 = answer_question_with_classifier(
#     rag_question_1,
#     domain_name="Customer_Support",
#     llm_client=llm_client,
#     embed_model=embed_model,
#     top_k=3
# )
# print(f"Answer (RAG): {result_rag_1['answer']}\n")
# print(f"Source: {result_rag_1['source']}")
# print("--------------------------------------------------")

In [ ]:
# rag_question_1 = "What is Meloni? Who is Modi"
# result_rag_1 = answer_question_with_classifier(
#     rag_question_1,
#     domain_name="NA",
#     llm_client=llm_client,
#     embed_model=embed_model,
#     top_k=3
# )
# print(f"Answer (RAG): {result_rag_1['answer']}\n")
# print(f"Source: {result_rag_1['source']}")
# print("--------------------------------------------------")

In [ ]:
# rag_question_1 = "What is Meloni? Who is Modi"
# result_rag_1 = answer_question_with_classifier(
#     rag_question_1,
#     domain_name="NA",
#     llm_client=llm_client,
#     embed_model=embed_model,
#     top_k=3
# )
# print(f"Answer (RAG): {result_rag_1['answer']}\n")
# print(f"Source: {result_rag_1['source']}")
# print("--------------------------------------------------")

In [ ]:
# rag_question_1 = "What is Meloni? Who is Modi"
# result_rag_1 = answer_question_with_classifier(
#     rag_question_1,
#     domain_name="NA",
#     llm_client=llm_client,
#     embed_model=embed_model,
#     top_k=3
# )
# print(f"Answer (RAG): {result_rag_1['answer']}\n")
# print(f"Source: {result_rag_1['source']}")
# print("--------------------------------------------------")

In [ ]:
# rag_question_1 = ragbench_by_domain["Customer_Support"]["techqa"]["test"]["question"][35]
# result_rag_1 = answer_question_with_classifier(
#     rag_question_1,
#     domain_name="Customer_Support",
#     llm_client=llm_client,
#     embed_model=embed_model,
#     top_k=3
# )
# print(f"Answer (RAG): {result_rag_1['answer']}\n")
# print(f"Source: {result_rag_1['source']}")
# print("--------------------------------------------------")

# # Test Case 1: RAG-suitable question (Customer Support domain)
# rag_question_1 = "How do I enter Ambient Mode on my TV?"
# result_rag_1 = answer_question_with_classifier(
#     rag_question_1,
#     domain_name="Customer_Support",
#     llm_client=llm_client,
#     embed_model=embed_model,
#     top_k=3
# )
# print(f"Answer (RAG): {result_rag_1['answer']}\n")
# print(f"Source: {result_rag_1['source']}")
# print("--------------------------------------------------")

# # Test Case 2: RAG-suitable question (General Knowledge domain)
# rag_question_2 = "When was the first mitrailleuse developed?"
# result_rag_2 = answer_question_with_classifier(
#     rag_question_2,
#     domain_name="General_Knowledge",
#     llm_client=llm_client,
#     embed_model=embed_model,
#     top_k=3
# )
# print(f"Answer (RAG): {result_rag_2['answer']}\n")
# print(f"Source: {result_rag_2['source']}")
# print("--------------------------------------------------")

# # Test Case 3: LLM-suitable general knowledge question
# llm_question_1 = "Who is the PM of India?"
# result_llm_1 = answer_question_with_classifier(
#     llm_question_1,
#     domain_name="General_Knowledge", # Domain doesn't matter for LLM-only questions but is required by the function signature
#     llm_client=llm_client
# )
# print(f"Answer (LLM): {result_llm_1['answer']}\n")
# print(f"Source: {result_llm_1['source']}")
# print("--------------------------------------------------")

# # Test Case 4: Another LLM-suitable general knowledge question
# llm_question_2 = "Who is Trump?"
# result_llm_2 = answer_question_with_classifier(
#     llm_question_2,
#     domain_name="NA",
#     llm_client=llm_client
# )
# print(f"Answer (LLM): {result_llm_2['answer']}\n")
# print(f"Source: {result_llm_2['source']}")
# print("--------------------------------------------------")

# # Test Case 5: A more specific LLM-suitable question
# llm_question_3 = "What is the capital of France?"
# result_llm_3 = answer_question_with_classifier(
#     llm_question_3,
#     domain_name="NA",
#     llm_client=llm_client
# )
# print(f"Answer (LLM): {result_llm_3['answer']}\n")
# print(f"Source: {result_llm_3['source']}")
# print("--------------------------------------------------")

# # Test Case 6: A question that might be ambiguous, favoring RAG due to keyword absence
# ambiguous_question = "What are the effects of high inflation?"
# result_ambiguous = answer_question_with_classifier(
#     ambiguous_question,
#     domain_name="Finance", # Relevant domain for this type of question
#     llm_client=llm_client,
#     embed_model=embed_model,
#     top_k=3
# )
# print(f"Answer (Ambiguous): {result_ambiguous['answer']}\n")
# print(f"Source: {result_ambiguous['source']}")
# print("--------------------------------------------------")

In [ ]:
def run_rag_sample(domain_name, dataset_name, sample_idx=0, top_k=5):
    # Get dataset
    domain_datasets = ragbench_by_domain[domain_name]
    dataset_data = domain_datasets[dataset_name]

    # Pick sample
    entry = dataset_data["test"][sample_idx]

    question = entry["question"]
    gt_response = entry["response"]
    gt_documents = entry.get("documents", [])

    print("\n" + "=" * 100)
    print(f"Domain   : {domain_name}")
    print(f"Dataset  : {dataset_name}")
    print(f"Sample # : {sample_idx}")
    print(f"Question : {question}")
    print("=" * 100)

    # ------------------------------------------------------------------
    # Ground Truth Documents
    # ------------------------------------------------------------------
    print("\n" + "=" * 100)
    print(f"GROUND TRUTH DOCUMENTS ({len(gt_documents)})")
    print("=" * 100)

    for i, doc in enumerate(gt_documents):
        print(f"\n[GT Document {i+1}]")
        print(doc[:1000])
        if len(doc) > 1000:
            print("...")

    # ------------------------------------------------------------------
    # Retrieval
    # ------------------------------------------------------------------
    retrieved_docs = retrieve(
        question,
        domain_name,
        top_k=top_k
    )

    retrieved_texts = []

    print("\n" + "=" * 100)
    print(f"RETRIEVED DOCUMENTS (Top-{top_k})")
    print("=" * 100)

    for i, doc in enumerate(retrieved_docs):

        doc_text = (
            doc["text"]
            if isinstance(doc, dict) and "text" in doc
            else str(doc)
        )

        retrieved_texts.append(doc_text)

        print(f"\n[Retrieved Document {i+1}]")
        print(doc_text[:1000])

        if len(doc_text) > 1000:
            print("...")

    # ------------------------------------------------------------------
    # Build Context
    # ------------------------------------------------------------------
    context = "\n\n".join(retrieved_texts)

    # ------------------------------------------------------------------
    # Generate Answer
    # ------------------------------------------------------------------
    answer = ask_rag(
        context,
        question,
        llm_client=llm_client
    )

    # ------------------------------------------------------------------
    # Results
    # ------------------------------------------------------------------
    print("\n" + "=" * 100)
    print("RAG RESPONSE")
    print("=" * 100)
    print(answer)

    print("\n" + "=" * 100)
    print("GROUND TRUTH RESPONSE")
    print("=" * 100)
    print(gt_response)

    # ------------------------------------------------------------------
    # Quick Stats
    # ------------------------------------------------------------------
    print("\n" + "=" * 100)
    print("SUMMARY")
    print("=" * 100)
    print(f"GT Documents        : {len(gt_documents)}")
    print(f"Retrieved Documents : {len(retrieved_texts)}")
    print(f"Question Length     : {len(question)}")
    print(f"GT Response Length  : {len(gt_response)}")
    print(f"RAG Response Length : {len(answer)}")

    return {
        "domain": domain_name,
        "dataset": dataset_name,
        "sample_idx": sample_idx,
        "question": question,
        "ground_truth_docs": gt_documents,
        "retrieved_docs": retrieved_docs,
        "retrieved_texts": retrieved_texts,
        "rag_response": answer,
        "ground_truth": gt_response,
    }

In [ ]:
# ── Tunable knobs ──────────────────────────────────────────────

#Generator - LLM
MODEL_NAME = LLM_CHOICES[0]
MODEL_NAME_BIG = LLM_CHOICES[3] #Judge LLM
HYDE_MODEL = LLM_CHOICES[0]

# ── Feature flags ──────────────────────────────────────────────
ENABLE_HYBRID     = True    # dense + BM25
ENABLE_HYDE       = True    # hypothetical document expansion
ENABLE_RERANKING  = True    # MonoT5 reranker
RERANKER_TYPE = "monot5"     # monot5 | tilde

# ── Tunable knobs ──────────────────────────────────────────────
RETRIEVE_TOP_K    = 10      # initial candidates from each retriever
RERANK_TOP_K      = 3       # final docs after reranking
HYBRID_ALPHA      = 0.5     # 0=BM25 only, 1=dense only, 0.5=balanced
MONOT5_MODEL      = "castorini/monot5-base-msmarco-10k"

# TILDE-style practical alternative
TILDE_MODEL = "BAAI/bge-reranker-base"

HYDE_LLM_MODEL = MODEL_NAME

# ── Prompt strategy feature flag ────────────────────────────────
PROMPT_STRATEGY = "short"   # "short" | "long" | "long_cot"

# ── Repacking feature flag ──────────────────────────────────────
ENABLE_REPACKING = True      # reorder for LLM attention
REPACK_STRATEGY  = "sides"   # "forward" | "reverse" | "sides"

# ── Feature flags ──────────────────────────────────────────────
ENABLE_SUMMARIZATION = True
SUMMARIZATION_TYPE   = "recomp"   # "recomp" | "longllmlingua"

# RECOMP config
RECOMP_TOP_K_SENTS      = 8
RECOMP_MIN_SCORE        = 0.10
RECOMP_GROUNDING_BOOST  = 0.15
RECOMP_MIN_KEEP_RATIO   = 0.60
RECOMP_KEEP_CRITICAL    = True

# LongLLMLingua config
LLMLINGUA_RATE     = 0.5          # compression rate (0.5 = 50% kept)
LLMLINGUA_MODEL    = "NousResearch/Llama-2-7b-hf"   # smaller works too

ENABLE_HYDE = True
ENABLE_HYBRID = True
ENABLE_RERANKING = True
RERANKER_TYPE = "monot5"   # or "tilde"

ENABLE_SUMMARIZATION = True
ENABLE_REPACKING = True



run_rag_sample("General_Knowledge", "hagrid", sample_idx=0)


In [ ]:
# ── Feature flags ──────────────────────────────────────────────
ENABLE_HYBRID     = True    # dense + BM25
ENABLE_HYDE       = False    # hypothetical document expansion
ENABLE_RERANKING  = True    # MonoT5 reranker
RERANKER_TYPE = "monot5"     # monot5 | tilde

# ── Tunable knobs ──────────────────────────────────────────────
RETRIEVE_TOP_K    = 10      # initial candidates from each retriever
RERANK_TOP_K      = 3       # final docs after reranking
HYBRID_ALPHA      = 0    # 0=BM25 only, 1=dense only, 0.5=balanced

# ── Repacking feature flag ──────────────────────────────────────
ENABLE_REPACKING = True      # reorder for LLM attention
REPACK_STRATEGY  = "sides"   # "forward" | "reverse" | "sides"

# ── Feature flags ──────────────────────────────────────────────
ENABLE_SUMMARIZATION = True
SUMMARIZATION_TYPE   = "recomp"   # "recomp" | "longllmlingua"
run_rag_sample("Finance", "finqa", sample_idx=0)

In [ ]:
# # ── Feature flags ──────────────────────────────────────────────
# ENABLE_HYBRID     = True    # dense + BM25
# ENABLE_HYDE       = False    # hypothetical document expansion
# ENABLE_RERANKING  = True    # MonoT5 reranker
# RERANKER_TYPE = "monot5"     # monot5 | tilde

# # ── Tunable knobs ──────────────────────────────────────────────
# RETRIEVE_TOP_K    = 10      # initial candidates from each retriever
# RERANK_TOP_K      = 3       # final docs after reranking
# HYBRID_ALPHA      = 1    # 0=BM25 only, 1=dense only, 0.5=balanced

# # ── Repacking feature flag ──────────────────────────────────────
# ENABLE_REPACKING = False      # reorder for LLM attention
# REPACK_STRATEGY  = "sides"   # "forward" | "reverse" | "sides"

# # ── Feature flags ──────────────────────────────────────────────
# ENABLE_SUMMARIZATION = False
# SUMMARIZATION_TYPE   = "recomp"   # "recomp" | "longllmlingua"
# run_rag_sample("Finance", "finqa", sample_idx=15)

In [ ]:
# # ── Feature flags ──────────────────────────────────────────────
# ENABLE_HYBRID     = True    # dense + BM25
# ENABLE_HYDE       = False    # hypothetical document expansion
# ENABLE_RERANKING  = True    # MonoT5 reranker
# RERANKER_TYPE = "monot5"     # monot5 | tilde

# # ── Tunable knobs ──────────────────────────────────────────────
# RETRIEVE_TOP_K    = 10      # initial candidates from each retriever
# RERANK_TOP_K      = 3       # final docs after reranking
# HYBRID_ALPHA      = 0.2    # 0=BM25 only, 1=dense only, 0.5=balanced

# # ── Repacking feature flag ──────────────────────────────────────
# ENABLE_REPACKING = False      # reorder for LLM attention
# REPACK_STRATEGY  = "sides"   # "forward" | "reverse" | "sides"

# # ── Feature flags ──────────────────────────────────────────────
# ENABLE_SUMMARIZATION = False
# SUMMARIZATION_TYPE   = "recomp"   # "recomp" | "longllmlingua"
# run_rag_sample("Finance", "finqa", sample_idx=10)

In [ ]:
#run_rag_sample("Bio_Medical", "pubmedqa", sample_idx=0)

In [ ]:
# # ── Feature flags ──────────────────────────────────────────────
# ENABLE_HYBRID     = True    # dense + BM25
# ENABLE_HYDE       = False    # hypothetical document expansion
# ENABLE_RERANKING  = True    # MonoT5 reranker
# RERANKER_TYPE = "monot5"     # monot5 | tilde

# # ── Tunable knobs ──────────────────────────────────────────────
# RETRIEVE_TOP_K    = 20      # initial candidates from each retriever
# RERANK_TOP_K      = 5       # final docs after reranking

# top_k = 5

# HYBRID_ALPHA      = 0.5    # 0=BM25 only, 1=dense only, 0.5=balanced

# # ── Repacking feature flag ──────────────────────────────────────
# ENABLE_REPACKING = False      # reorder for LLM attention
# REPACK_STRATEGY  = "sides"   # "forward" | "reverse" | "sides"

# # ── Feature flags ──────────────────────────────────────────────
# ENABLE_SUMMARIZATION = False
# SUMMARIZATION_TYPE   = "recomp"   # "recomp" | "longllmlingua"

# run_rag_sample("Bio_Medical", "pubmedqa", sample_idx=16)

In [ ]:
#run_rag_sample("Bio_Medical", "pubmedqa", sample_idx=15)

In [ ]:
run_rag_sample("Bio_Medical", "pubmedqa", sample_idx=18)

In [ ]:
#run_rag_sample("Legal_Contracts", "cuad", sample_idx=0)

In [ ]:
# HYBRID_ALPHA = 0.5
# run_rag_sample("Legal_Contracts", "cuad", sample_idx=37)

In [ ]:
#run_rag_sample("Legal_Contracts", "cuad", sample_idx=10)

In [ ]:
#run_rag_sample("Legal_Contracts", "cuad", sample_idx=15)

In [ ]:
#run_rag_sample("Legal_Contracts", "cuad", sample_idx=3)

In [ ]:
# HYBRID_ALPHA      = 0
# run_rag_sample("Legal_Contracts", "cuad", sample_idx=23)

In [ ]:

#run_rag_sample("Customer_Support", "delucionqa", sample_idx=0)

In [ ]:
#run_rag_sample("Customer_Support", "delucionqa", sample_idx=10)

# Stage 6 – Evaluation and Deployment

In [ ]:
def ask_judge_orig(prompt, llm_client):
  response = llm_client.chat.completions.create(
      model=MODEL_NAME_BIG,
      messages=[
          {"role": "system", "content": "You are a helpful RAG eval assistant"},
          {"role": "user", "content": prompt},
      ],
      temperature=0.3,
  )
  return response.choices[0].message.content

In [ ]:
def build_evaluation_prompt_long(documents_text, question, answer_text):
    return f"""
I asked someone to answer a question based on one or more documents.
Your task is to review their response and assess whether or not each sentence
in that response is supported by text in the documents. And if so, which
sentences in the documents provide that support. You will also tell me which
of the documents contain useful information for answering the question, and
which of the documents the answer was sourced from.
Here are the documents, each of which is split into sentences. Alongside each
sentence is associated key, such as ’0a.’ or ’0b.’ that you can use to refer
to it:
‘‘‘
{documents_text}
‘‘‘
The question was:
‘‘‘
{question}
‘‘‘
Here is their response, split into sentences. Alongside each sentence is
associated key, such as ’a.’ or ’b.’ that you can use to refer to it. Note
that these keys are unique to the response, and are not related to the keys
in the documents:
‘‘‘
{answer_text}
‘‘‘
You must respond with a JSON object matching this schema:
‘‘‘
{{
"relevance_explanation": string,
"all_relevant_sentence_keys": [string],
"overall_supported_explanation": string,
"overall_supported": boolean,
"sentence_support_information": [
{{
"response_sentence_key": string,
"explanation": string,
"supporting_sentence_keys": [string],
"fully_supported": boolean
}},
],
"all_utilized_sentence_keys": [string]
}}
‘‘‘
The relevance_explanation field is a string explaining which documents
contain useful information for answering the question. Provide a step-by-step
breakdown of information provided in the documents and how it is useful for
answering the question.
The all_relevant_sentence_keys field is a list of all document sentences keys
(e.g. ’0a’) that are revant to the question. Include every sentence that is
useful and relevant to the question, even if it was not used in the response,
or if only parts of the sentence are useful. Ignore the provided response when
making this judgement and base your judgement solely on the provided documents
and question. Omit sentences that, if removed from the document, would not
impact someone’s ability to answer the question.
The overall_supported_explanation field is a string explaining why the response
*as a whole* is or is not supported by the documents. In this field, provide a
step-by-step breakdown of the claims made in the response and the support (or
lack thereof) for those claims in the documents. Begin by assessing each claim
separately, one by one; don’t make any remarks about the response as a whole
until you have assessed all the claims in isolation.
The overall_supported field is a boolean indicating whether the response as a
whole is supported by the documents. This value should reflect the conclusion
you drew at the end of your step-by-step breakdown in overall_supported_explanation.
In the sentence_support_information field, provide information about the support
*for each sentence* in the response.
The sentence_support_information field is a list of objects, one for each sentence
in the response. Each object MUST have the following fields:- response_sentence_key: a string identifying the sentence in the response.
This key is the same as the one used in the response above.- explanation: a string explaining why the sentence is or is not supported by the
documents.- supporting_sentence_keys: keys (e.g. ’0a’) of sentences from the documents that
support the response sentence. If the sentence is not supported, this list MUST
be empty. If the sentence is supported, this list MUST contain one or more keys.
In special cases where the sentence is supported, but not by any specific sentence,
you can use the string "supported_without_sentence" to indicate that the sentence
is generally supported by the documents. Consider cases where the sentence is
expressing inability to answer the question due to lack of relevant information in
the provided contex as "supported_without_sentence". In cases where the sentence
is making a general statement (e.g. outlining the steps to produce an answer, or
summarizing previously stated sentences, or a transition sentence), use the
sting "general".In cases where the sentence is correctly stating a well-known fact,
like a mathematical formula, use the string "well_known_fact". In cases where the
sentence is performing numerical reasoning (e.g. addition, multiplication), use
the string "numerical_reasoning".- fully_supported: a boolean indicating whether the sentence is fully supported by
the documents.- This value should reflect the conclusion you drew at the end of your step-by-step
breakdown in explanation.- If supporting_sentence_keys is an empty list, then fully_supported must be false.
- Otherwise, use fully_supported to clarify whether everything in the response
sentence is fully supported by the document text indicated in supporting_sentence_keys
(fully_supported = true), or whether the sentence is only partially or incompletely
supported by that document text (fully_supported = false).
The all_utilized_sentence_keys field is a list of all sentences keys (e.g. ’0a’) that
were used to construct the answer. Include every sentence that either directly supported
the answer, or was implicitly used to construct the answer, even if it was not used
in its entirety. Omit sentences that were not used, and could have been removed from
the documents without affecting the answer.
You must respond with a valid JSON string. Use escapes for quotes, e.g. ‘\\"‘, and
newlines, e.g. ‘\\n‘. Do not write anything before or after the JSON string. Do not
wrap the JSON string in backticks like ‘‘‘ or ‘‘‘json.
As a reminder: your task is to review the response and assess which documents contain
useful information pertaining to the question, and how each sentence in the response
is supported by the text in the documents.\

""".strip()

In [ ]:
def build_evaluation_prompt(documents_text, question, answer_text):
    return f"""
Evaluate the RAG response using the provided documents.

Documents (sentence-keyed):
{documents_text}

Question:
{question}

Response (sentence-keyed):
{answer_text}

Return ONLY valid JSON in this exact structure:

{{
  "overall_supported": true,
  "all_relevant_sentence_keys": ["0_0"],
  "all_utilized_sentence_keys": ["0_0"],
  "sentence_support_information": [
    {{
      "response_sentence_key": "r_0",
      "supporting_sentence_keys": ["0_0"],
      "fully_supported": true
    }}
  ]
}}

Rules:

- Use only document sentence keys that are explicitly present in the Documents section.
- Never invent, modify, infer, or approximate document sentence keys.
- Document keys look like: 0_0, 0_1, 1_0, 1_1.
- Response keys look like: r_0, r_1, r_2.

- all_relevant_sentence_keys must contain only document sentence keys that directly help answer the question.
- Do not include irrelevant background information.
- If a document sentence is not useful for answering the question, do not include it.

- all_utilized_sentence_keys must contain only document sentence keys that were actually used to generate or support the response.
- Do not include document sentences that were retrieved but not used.

- Multiple document sentence keys may support the same response sentence.
- Include all supporting_sentence_keys necessary to justify a response sentence.
- Do not omit supporting evidence when it is required to fully support a claim.

- Be conservative when assigning support.
- A response sentence is fully_supported only if the supporting document sentences explicitly and directly support the entire claim.
- Do not use assumptions, interpretations, paraphrased facts, external knowledge, world knowledge, or inferred information.
- If any part of a response sentence is unsupported, set fully_supported to false.

- supporting_sentence_keys must be an empty list if a response sentence is unsupported.
- Every supporting_sentence_key must exist in the Documents section.

- overall_supported must be true only if every response sentence is fully supported by the documents.
- If any response sentence is unsupported or partially supported, overall_supported must be false.

- Return only valid JSON matching the required schema.
- Do not return markdown.
- Do not return explanations outside the JSON structure.
- Do not include code fences.
- Do not include <think> tags.
- Do not include any text before or after the JSON.

""".strip()


In [ ]:
# ───────────────────────────────────────────────────────────────
# Sentence-key helpers for judge evaluation
# ───────────────────────────────────────────────────────────────

def build_keyed_response(answer):
    """
    Convert generated answer into sentence-keyed response.

    Example:
        {
            "r_0": "First response sentence.",
            "r_1": "Second response sentence."
        }
    """

    keyed_response = {}

    for i, sent in enumerate(
        split_into_sentences(str(answer))
    ):
        sent = sent.strip()

        if not sent:
            continue

        keyed_response[f"r_{i}"] = sent

    return keyed_response


def build_sentence_keyed_docs(retrieved_docs):
    """
    Convert retrieved documents into sentence-keyed documents.

    Example:
        {
            "0_0": "First sentence from retrieved doc 0.",
            "0_1": "Second sentence from retrieved doc 0.",
            "1_0": "First sentence from retrieved doc 1."
        }

    These keys must match the judge prompt rules:
        Document keys: 0_0, 0_1, 1_0, 1_1
        Response keys: r_0, r_1, r_2
    """

    keyed_docs = {}

    for doc_idx, doc in enumerate(retrieved_docs):

        if isinstance(doc, dict):
            text = doc.get("text", "")
        else:
            text = str(doc)

        text = str(text).strip()

        if not text:
            continue

        for sent_idx, sent in enumerate(
            split_into_sentences(text)
        ):
            sent = sent.strip()

            if not sent:
                continue

            key = f"{doc_idx}_{sent_idx}"
            keyed_docs[key] = sent

    return keyed_docs

# # Helpers
# def build_keyed_response(answer):
#     return {
#         f"r_{i}": sent.strip()
#         for i, sent in enumerate(split_into_sentences(answer))
#         if sent.strip()
#     }


# def build_sentence_keyed_docs(retrieved_docs):
#     keyed_docs = {}

#     for doc_idx, doc in enumerate(retrieved_docs):

#         text = (
#             doc.get("text", "")
#             if isinstance(doc, dict)
#             else str(doc)
#         )

#         for sent_idx, sent in enumerate(split_into_sentences(text)):
#             sent = sent.strip()

#             if not sent:
#                 continue

#             key = f"{doc_idx}_{sent_idx}"
#             keyed_docs[key] = sent

#     return keyed_docs


# def build_keyed_response(answer):
#     return {
#         f"r_{i}": sent
#         for i, sent in enumerate(split_into_sentences(answer))
#     }

# def build_sentence_keyed_docs(retrieved_docs):
#     keyed_docs = {}
#     for doc_idx, doc in enumerate(retrieved_docs):
#         for sent_idx, sent in enumerate(split_into_sentences(doc)):
#             key = f"{doc_idx}_{sent_idx}"   # safe
#             keyed_docs[key] = sent
#     return keyed_docs



'''
def build_sentence_keyed_docs(retrieved_docs):
    keyed_docs = {}
    for doc_idx, doc in enumerate(retrieved_docs):
        for sent_idx, sent in enumerate(split_into_sentences(doc)):
            key = f"{doc_idx}{string.ascii_lowercase[sent_idx]}"
            keyed_docs[key] = sent
    return keyed_docs

def build_keyed_response(answer):
    return {
        string.ascii_lowercase[i]: sent
        for i, sent in enumerate(split_into_sentences(answer))
    }

  '''

In [ ]:
import time
import random
import re
import json


# ───────────────────────────────────────────────────────────────
# 2) Judge call with retry for 429 / 503 / 502 / 504
# ───────────────────────────────────────────────────────────────

def ask_judge(prompt, llm_client, max_retries=5):
    last_error = None

    for attempt in range(max_retries):
        try:
            response = llm_client.chat.completions.create(
                model=MODEL_NAME_BIG,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are a strict RAG evaluation judge. "
                            "Return ONLY valid JSON. "
                            "No reasoning. No markdown. No <think> tags. "
                            "Use only sentence keys explicitly provided in the prompt."
                        )
                    },
                    {"role": "user", "content": prompt},
                ],
                temperature=0.0,
                max_tokens=3000,
            )
            return response.choices[0].message.content

        except Exception as e:
            msg = str(e)
            last_error = e

            if "tokens per day" in msg.lower():
                raise RuntimeError(f"TPD_EXHAUSTED: {msg}")

            if "503" in msg or "over capacity" in msg.lower() or "internal_server_error" in msg:
                wait_sec = (2 ** attempt) + random.uniform(0.5, 1.5)
                print(
                    f"  ⚠️ Judge model over capacity. Retrying in {wait_sec:.1f}s "
                    f"(attempt {attempt+1}/{max_retries})..."
                )
                time.sleep(wait_sec)
                continue

            if "429" in msg or "rate_limit_exceeded" in msg:
                wait_sec = 2 ** attempt

                ms_match = re.search(r"try again in (\d+)ms", msg)
                s_match  = re.search(r"try again in ([\d.]+)s", msg)
                m_match  = re.search(r"try again in (\d+)m([\d.]+)s", msg)

                if ms_match:
                    wait_sec = int(ms_match.group(1)) / 1000.0
                elif s_match:
                    wait_sec = float(s_match.group(1))
                elif m_match:
                    wait_sec = int(m_match.group(1)) * 60 + float(m_match.group(2))

                wait_sec += random.uniform(0.1, 0.5)
                print(
                    f"  ⚠️ Judge rate limited. Retrying in {wait_sec:.1f}s "
                    f"(attempt {attempt+1}/{max_retries})..."
                )
                time.sleep(wait_sec)
                continue

            if "502" in msg or "504" in msg or "gateway" in msg.lower():
                wait_sec = (2 ** attempt) + random.uniform(0.5, 1.5)
                print(
                    f"  ⚠️ Gateway error. Retrying in {wait_sec:.1f}s "
                    f"(attempt {attempt+1}/{max_retries})..."
                )
                time.sleep(wait_sec)
                continue

            raise

    raise RuntimeError(f"Judge failed after {max_retries} retries: {last_error}")


# ───────────────────────────────────────────────────────────────
# 3) Robust JSON parser
# ───────────────────────────────────────────────────────────────

def parse_judge_json(raw_text):
    """
    Robust parser for judge JSON.
    Handles:
    - <think>...</think>
    - markdown fences
    - missing commas between objects
    - trailing commas
    - invalid backslash escapes
    """
    if raw_text is None:
        raise ValueError("Judge output is None")

    cleaned = str(raw_text).strip()

    if not cleaned:
        raise ValueError("Judge output is empty")

    # Remove real Qwen thinking blocks
    cleaned = re.sub(
        r"<think>.*?</think>",
        "",
        cleaned,
        flags=re.DOTALL | re.IGNORECASE
    ).strip()

    # Remove escaped thinking blocks if model returns escaped tags
    cleaned = re.sub(
        r"&lt;think&gt;.*?&lt;/think&gt;",
        "",
        cleaned,
        flags=re.DOTALL | re.IGNORECASE
    ).strip()

    # Remove markdown fences
    cleaned = cleaned.replace("```json", "").replace("```", "").strip()

    # Extract JSON object
    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start == -1 or end == -1 or end < start:
        raise ValueError(f"No JSON object found: {cleaned[:300]}")

    cleaned = cleaned[start:end + 1]

    def repair_json_string(s):
        # Missing comma between objects: } {
        s = re.sub(r"}\s*{", "}, {", s)

        # Missing comma between arrays: ] [
        s = re.sub(r"]\s*\[", "], [", s)

        # Missing comma between adjacent strings on new lines
        s = re.sub(r'"\s*\n\s*"', '",\n"', s)

        # Remove trailing commas before ] or }
        s = re.sub(r",\s*([}\]])", r"\1", s)

        # Fix invalid backslash escapes
        s = re.sub(r'(?<!\\)\\(?!["\\/bfnrtu])', r"\\\\", s)

        return s

    attempts = [
        cleaned,
        repair_json_string(cleaned),
    ]

    last_error = None

    for attempt_text in attempts:
        try:
            return json.loads(attempt_text)
        except json.JSONDecodeError as e:
            last_error = e

    raise ValueError(
        f"JSON repair failed: {last_error}\nContent:\n{cleaned[:500]}"
    )


# ───────────────────────────────────────────────────────────────
# 4) Local key validation — no LLM repair
# ───────────────────────────────────────────────────────────────

def validate_and_clean_judge_json(judge_json, keyed_docs, keyed_answer=None):
    """
    Deterministically validate judge output against known keys.

    This does NOT call another LLM.
    It removes invalid keys locally and returns diagnostics.
    """

    valid_doc_keys = set(keyed_docs.keys())
    valid_resp_keys = set(keyed_answer.keys()) if keyed_answer else set()

    relevant_raw = set(judge_json.get("all_relevant_sentence_keys", []))
    utilized_raw = set(judge_json.get("all_utilized_sentence_keys", []))

    # Keep only valid document keys
    relevant_clean = relevant_raw & valid_doc_keys
    utilized_clean = utilized_raw & valid_doc_keys

    invalid_relevant = relevant_raw - valid_doc_keys
    invalid_utilized = utilized_raw - valid_doc_keys

    cleaned_support_info = []
    invalid_supporting = set()
    invalid_response = set()

    for info in judge_json.get("sentence_support_information", []):
        response_key = info.get("response_sentence_key")
        supporting_keys_raw = set(info.get("supporting_sentence_keys", []))

        # Validate response key
        if keyed_answer and response_key not in valid_resp_keys:
            invalid_response.add(response_key)
            # Keep record, but mark as suspicious through diagnostics.

        # Clean supporting keys
        supporting_keys_clean = sorted(supporting_keys_raw & valid_doc_keys)
        invalid_supporting |= (supporting_keys_raw - valid_doc_keys)

        fully_supported = bool(info.get("fully_supported", False))

        # Important consistency fix:
        # If no valid evidence remains, sentence cannot be fully supported.
        if not supporting_keys_clean:
            fully_supported = False

        cleaned_support_info.append({
            "response_sentence_key": response_key,
            "supporting_sentence_keys": supporting_keys_clean,
            "fully_supported": fully_supported
        })

    cleaned_json = {
        "overall_supported": bool(judge_json.get("overall_supported", False)),
        "all_relevant_sentence_keys": sorted(relevant_clean),
        "all_utilized_sentence_keys": sorted(utilized_clean),
        "sentence_support_information": cleaned_support_info,
    }

    diagnostics = {
        "invalid_relevant_count": len(invalid_relevant),
        "invalid_utilized_count": len(invalid_utilized),
        "invalid_supporting_count": len(invalid_supporting),
        "invalid_response_count": len(invalid_response),

        "invalid_relevant_keys": sorted(invalid_relevant),
        "invalid_utilized_keys": sorted(invalid_utilized),
        "invalid_supporting_keys": sorted(invalid_supporting),
        "invalid_response_keys": sorted([k for k in invalid_response if k]),

        "raw_relevant_count": len(relevant_raw),
        "raw_utilized_count": len(utilized_raw),
        "clean_relevant_count": len(relevant_clean),
        "clean_utilized_count": len(utilized_clean),
        "valid_doc_key_count": len(valid_doc_keys),
    }

    return cleaned_json, diagnostics


# ───────────────────────────────────────────────────────────────
# 5) One-call robust judge without LLM repair
# ───────────────────────────────────────────────────────────────

def get_valid_judge_json(
    eval_prompt,
    llm_client,
    keyed_docs,
    keyed_answer,
    max_json_retries=3
):
    """
    Full judge workflow without repair LLM call:
    1. Ask judge
    2. Parse JSON
    3. Validate keys locally
    4. Remove invalid keys locally
    5. Return cleaned JSON + diagnostics
    """

    last_error = None

    for attempt in range(max_json_retries):
        try:
            raw = ask_judge(eval_prompt, llm_client=llm_client)
            judge_json = parse_judge_json(raw)

            cleaned_json, diagnostics = validate_and_clean_judge_json(
                judge_json=judge_json,
                keyed_docs=keyed_docs,
                keyed_answer=keyed_answer
            )

            total_invalid = (
                diagnostics["invalid_relevant_count"]
                + diagnostics["invalid_utilized_count"]
                + diagnostics["invalid_supporting_count"]
                + diagnostics["invalid_response_count"]
            )

            if total_invalid > 0:
                print(
                    f"  ⚠️ Judge invalid keys cleaned locally | "
                    f"relevant={diagnostics['invalid_relevant_count']}, "
                    f"utilized={diagnostics['invalid_utilized_count']}, "
                    f"supporting={diagnostics['invalid_supporting_count']}, "
                    f"response={diagnostics['invalid_response_count']}"
                )

            return cleaned_json, diagnostics

        except (ValueError, json.JSONDecodeError) as e:
            last_error = e
            print(
                f"  ⚠️ JSON parse failed "
                f"(attempt {attempt+1}/{max_json_retries})"
            )

            if attempt < max_json_retries - 1:
                eval_prompt = eval_prompt + (
                    "\n\nCRITICAL: Previous response had invalid JSON. "
                    "Return complete valid JSON only. "
                    "Ensure commas are correctly placed between objects in arrays."
                )
                time.sleep(1)

    raise ValueError(f"Judge JSON parsing failed after retries: {last_error}")


print("Judge functions loaded ✅")

In [ ]:
# import time
# import random
# import re


# # ───────────────────────────────────────────────────────────────
# # 2) Judge call with retry for 429 / 503 / 502 / 504
# # ───────────────────────────────────────────────────────────────

# def ask_judge(prompt, llm_client, max_retries=5):
#     last_error = None

#     for attempt in range(max_retries):
#         try:
#             response = llm_client.chat.completions.create(
#                 model=MODEL_NAME_BIG,
#                 messages=[
#                     {
#                         "role": "system",
#                         "content": (
#                             "You are a strict RAG evaluation judge. "
#                             "Return ONLY valid JSON. "
#                             "No reasoning. No markdown. No <think> tags. "
#                             "Use only sentence keys explicitly provided in the prompt."
#                         )
#                     },
#                     {"role": "user", "content": prompt},
#                 ],
#                 temperature=0.0,
#                 max_tokens=3000,
#             )
#             return response.choices[0].message.content

#         except Exception as e:
#             msg = str(e)
#             last_error = e

#             if "tokens per day" in msg.lower():
#                 raise RuntimeError(f"TPD_EXHAUSTED: {msg}")

#             if "503" in msg or "over capacity" in msg.lower() or "internal_server_error" in msg:
#                 wait_sec = (2 ** attempt) + random.uniform(0.5, 1.5)
#                 print(f"  ⚠️ Judge model over capacity. Retrying in {wait_sec:.1f}s "
#                       f"(attempt {attempt+1}/{max_retries})...")
#                 time.sleep(wait_sec)
#                 continue

#             if "429" in msg or "rate_limit_exceeded" in msg:
#                 wait_sec = 2 ** attempt

#                 ms_match = re.search(r"try again in (\d+)ms", msg)
#                 s_match  = re.search(r"try again in ([\d.]+)s", msg)
#                 m_match  = re.search(r"try again in (\d+)m([\d.]+)s", msg)

#                 if ms_match:
#                     wait_sec = int(ms_match.group(1)) / 1000.0
#                 elif s_match:
#                     wait_sec = float(s_match.group(1))
#                 elif m_match:
#                     wait_sec = int(m_match.group(1)) * 60 + float(m_match.group(2))

#                 wait_sec += random.uniform(0.1, 0.5)
#                 print(f"  ⚠️ Judge rate limited. Retrying in {wait_sec:.1f}s "
#                       f"(attempt {attempt+1}/{max_retries})...")
#                 time.sleep(wait_sec)
#                 continue

#             if "502" in msg or "504" in msg or "gateway" in msg.lower():
#                 wait_sec = (2 ** attempt) + random.uniform(0.5, 1.5)
#                 print(f"  ⚠️ Gateway error. Retrying in {wait_sec:.1f}s "
#                       f"(attempt {attempt+1}/{max_retries})...")
#                 time.sleep(wait_sec)
#                 continue

#             raise

#     raise RuntimeError(f"Judge failed after {max_retries} retries: {last_error}")


# # ───────────────────────────────────────────────────────────────
# # 3) Robust JSON parser
# # ───────────────────────────────────────────────────────────────

# def parse_judge_json(raw_text):
#     """
#     Robust parser for judge JSON.
#     Handles:
#     - <think>...</think>
#     - markdown fences
#     - missing commas between objects
#     - trailing commas
#     - invalid backslash escapes
#     """
#     if raw_text is None:
#         raise ValueError("Judge output is None")

#     cleaned = str(raw_text).strip()

#     if not cleaned:
#         raise ValueError("Judge output is empty")

#     # Remove Qwen thinking blocks
#     cleaned = re.sub(r"<think>.*?</think>", "", cleaned, flags=re.DOTALL).strip()

#     # Remove markdown fences
#     cleaned = cleaned.replace("```json", "").replace("```", "").strip()

#     # Extract JSON object
#     start = cleaned.find("{")
#     end = cleaned.rfind("}")

#     if start == -1 or end == -1 or end < start:
#         raise ValueError(f"No JSON object found: {cleaned[:300]}")

#     cleaned = cleaned[start:end + 1]

#     def repair_json_string(s):
#         # Missing comma between objects: } {
#         s = re.sub(r"}\s*{", "}, {", s)

#         # Missing comma between arrays: ] [
#         s = re.sub(r"]\s*\[", "], [", s)

#         # Missing comma between adjacent strings on new lines
#         s = re.sub(r'"\s*\n\s*"', '",\n"', s)

#         # Remove trailing commas before ] or }
#         s = re.sub(r",\s*([}\]])", r"\1", s)

#         # Fix invalid backslash escapes
#         s = re.sub(r'(?<!\\)\\(?!["\\/bfnrtu])', r"\\\\", s)

#         return s

#     attempts = [
#         cleaned,
#         repair_json_string(cleaned),
#     ]

#     last_error = None

#     for attempt_text in attempts:
#         try:
#             return json.loads(attempt_text)
#         except json.JSONDecodeError as e:
#             last_error = e

#     raise ValueError(f"JSON repair failed: {last_error}\nContent:\n{cleaned[:500]}")

# # ───────────────────────────────────────────────────────────────
# # 4) Local key validation — no LLM repair
# # ───────────────────────────────────────────────────────────────

# def validate_and_clean_judge_json(judge_json, keyed_docs, keyed_answer=None):
#     """
#     Deterministically validate judge output against known keys.

#     This does NOT call another LLM.
#     It removes invalid keys locally and returns diagnostics.
#     """

#     valid_doc_keys = set(keyed_docs.keys())
#     valid_resp_keys = set(keyed_answer.keys()) if keyed_answer else set()

#     relevant_raw = set(judge_json.get("all_relevant_sentence_keys", []))
#     utilized_raw = set(judge_json.get("all_utilized_sentence_keys", []))

#     # Keep only valid document keys
#     relevant_clean = relevant_raw & valid_doc_keys
#     utilized_clean = utilized_raw & valid_doc_keys

#     invalid_relevant = relevant_raw - valid_doc_keys
#     invalid_utilized = utilized_raw - valid_doc_keys

#     cleaned_support_info = []
#     invalid_supporting = set()
#     invalid_response = set()

#     for info in judge_json.get("sentence_support_information", []):
#         response_key = info.get("response_sentence_key")
#         supporting_keys_raw = set(info.get("supporting_sentence_keys", []))

#         # Validate response key
#         if keyed_answer and response_key not in valid_resp_keys:
#             invalid_response.add(response_key)
#             # Keep the record, but response key is suspicious.
#             # You may also choose to skip it entirely.

#         # Clean supporting keys
#         supporting_keys_clean = sorted(supporting_keys_raw & valid_doc_keys)
#         invalid_supporting |= (supporting_keys_raw - valid_doc_keys)

#         cleaned_support_info.append({
#             "response_sentence_key": response_key,
#             "supporting_sentence_keys": supporting_keys_clean,
#             "fully_supported": bool(info.get("fully_supported", False))
#         })

#     cleaned_json = {
#         "overall_supported": bool(judge_json.get("overall_supported", False)),
#         "all_relevant_sentence_keys": sorted(relevant_clean),
#         "all_utilized_sentence_keys": sorted(utilized_clean),
#         "sentence_support_information": cleaned_support_info,
#     }

#     diagnostics = {
#         "invalid_relevant_count": len(invalid_relevant),
#         "invalid_utilized_count": len(invalid_utilized),
#         "invalid_supporting_count": len(invalid_supporting),
#         "invalid_response_count": len(invalid_response),

#         "invalid_relevant_keys": sorted(invalid_relevant),
#         "invalid_utilized_keys": sorted(invalid_utilized),
#         "invalid_supporting_keys": sorted(invalid_supporting),
#         "invalid_response_keys": sorted([k for k in invalid_response if k]),

#         "raw_relevant_count": len(relevant_raw),
#         "raw_utilized_count": len(utilized_raw),
#         "clean_relevant_count": len(relevant_clean),
#         "clean_utilized_count": len(utilized_clean),
#         "valid_doc_key_count": len(valid_doc_keys),
#     }

#     return cleaned_json, diagnostics

# # ───────────────────────────────────────────────────────────────
# # 5) One-call robust judge without LLM repair
# # ───────────────────────────────────────────────────────────────

# def get_valid_judge_json(
#     eval_prompt,
#     llm_client,
#     keyed_docs,
#     keyed_answer,
#     max_json_retries=3
# ):
#     """
#     Full judge workflow without repair LLM call:
#     1. Ask judge
#     2. Parse JSON
#     3. Validate keys locally
#     4. Remove invalid keys locally
#     5. Return cleaned JSON + diagnostics
#     """

#     last_error = None

#     for attempt in range(max_json_retries):
#         try:
#             raw = ask_judge(eval_prompt, llm_client=llm_client)
#             judge_json = parse_judge_json(raw)

#             cleaned_json, diagnostics = validate_and_clean_judge_json(
#                 judge_json=judge_json,
#                 keyed_docs=keyed_docs,
#                 keyed_answer=keyed_answer
#             )

#             total_invalid = (
#                 diagnostics["invalid_relevant_count"]
#                 + diagnostics["invalid_utilized_count"]
#                 + diagnostics["invalid_supporting_count"]
#                 + diagnostics["invalid_response_count"]
#             )

#             if total_invalid > 0:
#                 print(
#                     f"  ⚠️ Judge invalid keys cleaned locally | "
#                     f"relevant={diagnostics['invalid_relevant_count']}, "
#                     f"utilized={diagnostics['invalid_utilized_count']}, "
#                     f"supporting={diagnostics['invalid_supporting_count']}, "
#                     f"response={diagnostics['invalid_response_count']}"
#                 )

#             return cleaned_json, diagnostics

#         except (ValueError, json.JSONDecodeError) as e:
#             last_error = e
#             print(f"  ⚠️ JSON parse failed (attempt {attempt+1}/{max_json_retries})")

#             if attempt < max_json_retries - 1:
#                 eval_prompt = eval_prompt + (
#                     "\n\nCRITICAL: Previous response had invalid JSON. "
#                     "Return complete valid JSON only. "
#                     "Ensure commas are correctly placed between objects in arrays."
#                 )
#                 time.sleep(1)

#     raise ValueError(f"Judge JSON parsing failed after retries: {last_error}")



In [ ]:
# ───────────────────────────────────────────────────────────────
# 6) Safe metric computation
# ───────────────────────────────────────────────────────────────

def evaluate_ragbench_json(judge_json, keyed_docs):
    """
    Safe metric computation.

    Uses keyed_docs as source of truth.
    Guarantees all predicted scores are within [0, 1].
    """

    valid_doc_keys = set(keyed_docs.keys())

    relevant = set(judge_json.get("all_relevant_sentence_keys", [])) & valid_doc_keys
    utilized = set(judge_json.get("all_utilized_sentence_keys", [])) & valid_doc_keys

    overlap = relevant & utilized
    total = len(valid_doc_keys)

    relevance_score = len(relevant) / total if total else 0.0
    utilization_score = len(utilized) / total if total else 0.0
    completeness_score = len(overlap) / len(relevant) if relevant else 0.0

    relevance_score = float(np.clip(relevance_score, 0.0, 1.0))
    utilization_score = float(np.clip(utilization_score, 0.0, 1.0))
    completeness_score = float(np.clip(completeness_score, 0.0, 1.0))

    adherence_score = int(bool(judge_json.get("overall_supported", False)))

    return {
        "adherence_score": adherence_score,
        "hallucination_flag": 1 - adherence_score,

        "relevance_score": relevance_score,
        "utilization_score": utilization_score,
        "completeness_score": completeness_score,

        "num_valid_doc_keys": len(valid_doc_keys),
        "num_relevant_valid": len(relevant),
        "num_utilized_valid": len(utilized),
    }



In [ ]:
# ── Single-sample evaluation (debug / spot-check) ─────────────
def evaluate_sample(
    ragbench_by_domain,
    domain_name,
    dataset_name,
    llm_client,
    idx=0,
    top_k=5,
    split="test",
    verbose=True
):
    """
    Run retrieval + RAG answer + judge on ONE sample.

    Returns a dict with:
    - ground truth response
    - generated RAG answer
    - gold scores
    - predicted scores
    - retrieved docs
    - context
    - keyed docs
    - keyed answer
    - judge JSON
    - judge diagnostics

    Useful for debugging:
    - retrieval quality
    - prompt quality
    - generated answer vs ground truth answer
    - invalid judge keys
    - sentence-key alignment
    """

    # ── 0) Load sample ─────────────────────────────────────────
    entry = ragbench_by_domain[domain_name][dataset_name][split][idx]

    question = entry["question"]
    gt_response = entry.get("response", "")

    # ── 1) Retrieve ────────────────────────────────────────────
    retrieved_docs = retrieve(
        question,
        domain_name,
        embed_model=embed_model,
        llm_client=llm_client,
        top_k=top_k,
    )

    docs_text = [
        d["text"] if isinstance(d, dict) else d
        for d in retrieved_docs
    ]

    context = "\n\n".join(docs_text)

    # ── 2) Generate RAG answer ─────────────────────────────────
    rag_answer = ask_rag(
        context,
        question,
        llm_client=llm_client
    )

    # ── 3) Build keyed docs and keyed answer ───────────────────
    keyed_docs = build_sentence_keyed_docs(docs_text)

    documents_text = "\n".join(
        f"{k}. {v}" for k, v in keyed_docs.items()
    )

    keyed_answer = build_keyed_response(rag_answer)

    answer_text = "\n".join(
        f"{k}. {v}" for k, v in keyed_answer.items()
    )

    # Diagnostic only.
    # For scoring, use len(keyed_docs), not count_total_sentences(context).
    total_doc_sents_regex = count_total_sentences(context)
    total_doc_sents_keyed = len(keyed_docs)

    # ── 4) Ask judge using robust judge pipeline ───────────────
    eval_prompt = build_evaluation_prompt(
        documents_text,
        question,
        answer_text
    )

    judge_json, judge_diag = get_valid_judge_json(
        eval_prompt=eval_prompt,
        llm_client=llm_client,
        keyed_docs=keyed_docs,
        keyed_answer=keyed_answer,
        max_json_retries=3
    )

    # ── 5) Compute safe predicted metrics ──────────────────────
    pred = evaluate_ragbench_json(
        judge_json=judge_json,
        keyed_docs=keyed_docs
    )

    # ── 6) Package output ──────────────────────────────────────
    out = {
        "idx": idx,
        "domain": domain_name,
        "dataset": dataset_name,
        "split": split,

        "question": question,
        "ground_truth_response": gt_response,
        "rag_answer": rag_answer,

        "retrieved_docs": retrieved_docs,
        "docs_text": docs_text,
        "context": context,

        "keyed_docs": keyed_docs,
        "keyed_answer": keyed_answer,

        "documents_text": documents_text,
        "answer_text": answer_text,

        "judge_json": judge_json,
        "judge_diag": judge_diag,

        "total_doc_sents_regex": total_doc_sents_regex,
        "total_doc_sents_keyed": total_doc_sents_keyed,

        "gold": {
            "adherence":    entry.get("adherence_score", np.nan),
            "relevance":    entry.get("relevance_score", np.nan),
            "utilization":  entry.get("utilization_score", np.nan),
            "completeness": entry.get("completeness_score", np.nan),
        },

        "pred": pred,
    }

    # ── 7) Verbose debug print ─────────────────────────────────
    if verbose:
        print(f"── Sample {idx} ──")
        print(f"Domain/Dataset: {domain_name}/{dataset_name}")
        print(f"Question:\n{question}\n")

        print("Ground Truth Response:")
        print(gt_response)
        print()

        print("Generated RAG Answer:")
        print(rag_answer)
        print()

        print("Retrieved docs:")
        for i, d in enumerate(retrieved_docs, 1):
            text = d["text"] if isinstance(d, dict) else d

            if isinstance(d, dict):
                score = d.get("rerank_score", d.get("score", None))
            else:
                score = None

            score_text = (
                f" | score={score:.4f}"
                if isinstance(score, (int, float))
                else ""
            )

            print(f"  [{i}]{score_text} {text[:250]}...")
        print()

        print("Sentence counts:")
        print(f"  Regex count : {total_doc_sents_regex}")
        print(f"  Keyed count : {total_doc_sents_keyed}")
        print()

        print("Judge diagnostics:")
        print(f"  invalid_relevant_count   : {judge_diag.get('invalid_relevant_count', 0)}")
        print(f"  invalid_utilized_count   : {judge_diag.get('invalid_utilized_count', 0)}")
        print(f"  invalid_supporting_count : {judge_diag.get('invalid_supporting_count', 0)}")
        print(f"  invalid_response_count   : {judge_diag.get('invalid_response_count', 0)}")
        print()

        print("Judge JSON:")
        print(json.dumps(judge_json, indent=2))
        print()

        print(f"Gold: {out['gold']}")
        print(f"Pred: {out['pred']}")

    return out

In [ ]:
result = evaluate_sample(
    ragbench_by_domain,
    domain_name="Customer_Support",
    dataset_name="emanual",            # whichever subset
    llm_client=llm_client,
    idx=0,                          # change to inspect different samples
    top_k=3,
)

# Access pieces
result["judge_json"]
result["pred"]["adherence_score"]
result["gold"]

In [ ]:
result = evaluate_sample(
    ragbench_by_domain=ragbench_by_domain,
    domain_name="Finance",
    dataset_name="finqa",
    idx=0,
    split="test",
    llm_client=llm_client,
    top_k=3,
    verbose=True
)

result["judge_diag"]
result["pred"]
result["total_doc_sents_regex"]
result["total_doc_sents_keyed"]


In [ ]:
def ask_judge_for_sample(
    ragbench_by_domain,
    domain_name,
    dataset_name,
    sample_idx=0,
    split="test",
    llm_client=None,
    use_gt_documents=True,
    top_k=3
):
    """
    Generic helper to evaluate any sample from any domain/dataset.

    Parameters
    ----------
    ragbench_by_domain : dict
        Structure: ragbench_by_domain[domain][dataset][split]
    domain_name : str
        Example: "Customer_Support"
    dataset_name : str
        Example: "emanual"
    sample_idx : int
        Which sample to use
    split : str
        "train" / "test" / "validation"
    llm_client : object
        Your Groq/OpenAI client
    use_gt_documents : bool
        If True, use sample["documents"] from dataset
        If False, retrieve documents using your retrieve(query, domain_name, top_k)
    top_k : int
        Used only when use_gt_documents=False
    """

    # ---------------------------
    # 1. Fetch sample
    # ---------------------------
    sample = ragbench_by_domain[domain_name][dataset_name][split][sample_idx]

    question = sample["question"]
    answer = sample["response"]   # ground-truth answer from dataset

    # ---------------------------
    # 2. Select documents
    # ---------------------------
    if use_gt_documents:
        retrieved_docs = sample["documents"]
    else:
        retrieved_docs = retrieve(question, domain_name, top_k=top_k)

        # If your retrieve() returns dicts like {"text": ..., "doc_id": ...}
        if retrieved_docs and isinstance(retrieved_docs[0], dict):
            retrieved_docs = [doc["text"] for doc in retrieved_docs]

    # ---------------------------
    # 3. Build keyed docs
    # ---------------------------
    keyed_docs = build_sentence_keyed_docs(retrieved_docs)
    documents_text = "\n".join(f"{k}. {v}" for k, v in keyed_docs.items())

    # ---------------------------
    # 4. Build keyed answer
    # ---------------------------
    keyed_answer = build_keyed_response(answer)
    answer_text = "\n".join(f"{k}. {v}" for k, v in keyed_answer.items())

    # ---------------------------
    # 5. Build evaluation prompt
    # ---------------------------
    eval_prompt = build_evaluation_prompt(
        documents_text=documents_text,
        question=question,
        answer_text=answer_text
    )

    # ---------------------------
    # 6. Ask judge
    # ---------------------------
    judge_output = ask_judge(eval_prompt, llm_client=llm_client)

    return {
        "domain_name": domain_name,
        "dataset_name": dataset_name,
        "split": split,
        "sample_idx": sample_idx,
        "question": question,
        "answer": answer,
        "retrieved_docs": retrieved_docs,
        "keyed_docs": keyed_docs,
        "keyed_answer": keyed_answer,
        "eval_prompt": eval_prompt,
        "judge_output": judge_output
    }


In [ ]:
# result = ask_judge_for_sample(
#     ragbench_by_domain=ragbench_by_domain,
#     domain_name="Customer_Support",
#     dataset_name="emanual",
#     sample_idx=0,
#     split="test",
#     llm_client=llm_client,
#     use_gt_documents=False,
#     top_k=3
# )

# print(result["judge_output"])

In [ ]:
# result = ask_judge_for_sample(
#     ragbench_by_domain=ragbench_by_domain,
#     domain_name="General_Knowledge",
#     dataset_name="hagrid",
#     sample_idx=10,
#     split="test",
#     llm_client=llm_client,
#     use_gt_documents=True,
#     top_k=3
# )

# print(result["judge_output"])


In [ ]:
# result = ask_judge_for_sample(
#     ragbench_by_domain=ragbench_by_domain,
#     domain_name="General_Knowledge",
#     dataset_name="hagrid",
#     sample_idx=10,
#     split="test",
#     llm_client=llm_client,
#     use_gt_documents=False,
#     top_k=3
# )

# print(result["judge_output"])


#Ablation

In [ ]:
# @title Ablation: Dense vs Reranker vs Hybrid vs Summarization vs Repacking vs Full

import json
import re
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score


# ── Single-mode evaluation loop ───────────────────────────────
def evaluate_mode(
    ragbench_by_domain,
    domain_name,
    dataset_name,
    llm_client,
    top_k=3,
    max_samples=None,
    split="test",
    sleep_sec=1.0,
    save_intermediates=False
):
    """
    Evaluate one pipeline mode over N samples.

    Updated to use:
    - keyed_docs-based scoring
    - robust judge JSON parsing
    - local invalid-key cleanup
    - judge diagnostics
    """

    test_data = ragbench_by_domain[domain_name][dataset_name][split]
    n = min(len(test_data), max_samples) if max_samples else len(test_data)

    results = []

    for idx in range(n):
        try:
            entry = test_data[idx]
            question = entry["question"]
            gt_response = entry.get("response", "")

            # ── 1) Retrieve ───────────────────────────────────
            retrieved_docs = retrieve(
                question,
                domain_name,
                embed_model=embed_model,
                llm_client=llm_client,
                top_k=top_k
            )

            docs_text = [
                d["text"] if isinstance(d, dict) else d
                for d in retrieved_docs
            ]

            context = "\n\n".join(docs_text)

            # ── 2) Generate RAG answer ────────────────────────
            rag_answer = ask_rag(
                context,
                question,
                llm_client=llm_client
            )

            # ── 3) Build keyed docs and keyed answer ──────────
            keyed_docs = build_sentence_keyed_docs(docs_text)

            documents_text = "\n".join(
                f"{k}. {v}" for k, v in keyed_docs.items()
            )

            keyed_answer = build_keyed_response(rag_answer)

            answer_text = "\n".join(
                f"{k}. {v}" for k, v in keyed_answer.items()
            )

            # Diagnostic only. Do not use this for scoring.
            total_doc_sents_regex = count_total_sentences(context)
            total_doc_sents_keyed = len(keyed_docs)

            # ── 4) Judge ──────────────────────────────────────
            eval_prompt = build_evaluation_prompt(
                documents_text,
                question,
                answer_text
            )

            judge_json, judge_diag = get_valid_judge_json(
                eval_prompt=eval_prompt,
                llm_client=llm_client,
                keyed_docs=keyed_docs,
                keyed_answer=keyed_answer,
                max_json_retries=3
            )

            # ── 5) Safe predicted metrics ─────────────────────
            pred = evaluate_ragbench_json(
                judge_json=judge_json,
                keyed_docs=keyed_docs
            )

            row = {
                "idx": idx,

                # Optional debug fields
                "question": question,
                "ground_truth_response": gt_response,
                "rag_answer": rag_answer,

                # Gold scores
                "gold_adherence": entry.get("adherence_score", np.nan),
                "gold_relevance": entry.get("relevance_score", np.nan),
                "gold_utilization": entry.get("utilization_score", np.nan),
                "gold_completeness": entry.get("completeness_score", np.nan),

                # Predicted scores
                "pred_adherence": pred["adherence_score"],
                "pred_relevance": pred["relevance_score"],
                "pred_utilization": pred["utilization_score"],
                "pred_completeness": pred["completeness_score"],

                # Diagnostics
                "total_doc_sents_regex": total_doc_sents_regex,
                "total_doc_sents_keyed": total_doc_sents_keyed,

                "invalid_relevant_count": judge_diag.get("invalid_relevant_count", 0),
                "invalid_utilized_count": judge_diag.get("invalid_utilized_count", 0),
                "invalid_supporting_count": judge_diag.get("invalid_supporting_count", 0),
                "invalid_response_count": judge_diag.get("invalid_response_count", 0),

                "raw_relevant_count": judge_diag.get("raw_relevant_count", np.nan),
                "raw_utilized_count": judge_diag.get("raw_utilized_count", np.nan),
                "clean_relevant_count": judge_diag.get("clean_relevant_count", np.nan),
                "clean_utilized_count": judge_diag.get("clean_utilized_count", np.nan),
                "valid_doc_key_count": judge_diag.get("valid_doc_key_count", np.nan),
            }

            results.append(row)

            if sleep_sec:
                time.sleep(sleep_sec)

            if (idx + 1) % 10 == 0:
                print(f"  Processed {idx + 1}/{n}")

        except RuntimeError as e:
            if "TPD_EXHAUSTED" in str(e):
                print(f"  Stopping at sample {idx}: daily token limit hit")
                break

            print(f"  Failed sample {idx}: {e}")

        except Exception as e:
            print(f"  Failed sample {idx}: {e}")
            continue

    return pd.DataFrame(results)


# ── RMSE + AUROC ───────────────────────────────────────────────
def rmse(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)

    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)

    y_true = y_true[mask]
    y_pred = y_pred[mask]

    return np.sqrt(np.mean((y_pred - y_true) ** 2)) if len(y_true) else np.nan

def auroc(y_true, y_score):
    y_true = np.array(y_true, dtype=float)
    y_score = np.array(y_score, dtype=float)

    mask = ~np.isnan(y_true) & ~np.isnan(y_score)

    y_true = y_true[mask]
    y_score = y_score[mask]

    if len(y_true) == 0:
        return np.nan

    return roc_auc_score(y_true, y_score) if len(np.unique(y_true)) >= 2 else np.nan


# def auroc(y_true, y_score):
#     y_true = np.array(y_true)
#     y_score = np.array(y_score, dtype=float)

#     mask = ~np.isnan(y_score)

#     y_true = y_true[mask]
#     y_score = y_score[mask]

#     return roc_auc_score(y_true, y_score) if len(np.unique(y_true)) >= 2 else np.nan


def compute_metrics(df):
    """
    Compute final metrics safely.

    Important:
    - Clips predicted scores to [0, 1]
    - Prevents old bad rows from causing RMSE > 1
    - AUROC returns NaN if only one gold class exists
    """

    if df.empty:
        return {
            "Hallucination AUROC": np.nan,
            "Relevance RMSE": np.nan,
            "Utilization RMSE": np.nan,
            "Completeness RMSE": np.nan,
        }

    df = df.copy()

    # Safety: ensure predicted continuous scores are bounded
    for col in ["pred_relevance", "pred_utilization", "pred_completeness"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").clip(0.0, 1.0)

    df["pred_adherence"] = pd.to_numeric(df["pred_adherence"], errors="coerce").clip(0.0, 1.0)

    return {
        "Hallucination AUROC": auroc(
            1 - df["gold_adherence"].astype(int),
            1 - df["pred_adherence"].astype(float)
        ),
        "Relevance RMSE": rmse(
            df["gold_relevance"],
            df["pred_relevance"]
        ),
        "Utilization RMSE": rmse(
            df["gold_utilization"],
            df["pred_utilization"]
        ),
        "Completeness RMSE": rmse(
            df["gold_completeness"],
            df["pred_completeness"]
        ),
    }

# ── Run 6-mode ablation ────────────────────────────────────────
def run_ablation(
    ragbench_by_domain,
    domain_name,
    dataset_name,
    llm_client,
    top_k=3,
    max_samples=None,
    split="test",
    sleep_sec=1.0
):
    global ENABLE_HYDE, ENABLE_HYBRID, ENABLE_RERANKING
    global ENABLE_SUMMARIZATION, SUMMARIZATION_TYPE
    global ENABLE_REPACKING, REPACK_STRATEGY

    # Save current global flag state so notebook is restored after run
    old_flags = {
        "ENABLE_HYDE": ENABLE_HYDE,
        "ENABLE_HYBRID": ENABLE_HYBRID,
        "ENABLE_RERANKING": ENABLE_RERANKING,
        "ENABLE_SUMMARIZATION": ENABLE_SUMMARIZATION,
        "SUMMARIZATION_TYPE": SUMMARIZATION_TYPE,
        "ENABLE_REPACKING": ENABLE_REPACKING,
        "REPACK_STRATEGY": REPACK_STRATEGY,
    }

    modes = [
        (
            "Dense (baseline)",
            {
                "hyde": False,
                "hybrid": False,
                "rerank": False,
                "summarize": False,
                "summ_type": "recomp",
                "repack": False,
                "repack_strat": "forward",
            },
        ),

        (
            "+ Reranker",
            {
                "hyde": False,
                "hybrid": False,
                "rerank": True,
                "summarize": False,
                "summ_type": "recomp",
                "repack": False,
                "repack_strat": "forward",
            },
        ),

        (
            "Hybrid + Reranker",
            {
                "hyde": False,
                "hybrid": True,
                "rerank": True,
                "summarize": False,
                "summ_type": "recomp",
                "repack": False,
                "repack_strat": "forward",
            },
        ),

        (
            "+ Summarization (RECOMP)",
            {
                "hyde": False,
                "hybrid": True,
                "rerank": True,
                "summarize": True,
                "summ_type": "recomp",
                "repack": False,
                "repack_strat": "forward",
            },
        ),

        (
            "+ Repacking (sides)",
            {
                "hyde": False,
                "hybrid": True,
                "rerank": True,
                "summarize": True,          # important: keep summarization enabled
                "summ_type": "recomp",
                "repack": True,
                "repack_strat": "sides",
            },
        ),

        (
            "Full pipeline (+ HyDE)",
            {
                "hyde": True,
                "hybrid": True,
                "rerank": True,
                "summarize": True,
                "summ_type": "recomp",
                "repack": True,
                "repack_strat": "sides",
            },
        ),
    ]

    ablation = {}
    mode_dfs = {}

    try:
        for mode_name, flags in modes:
            ENABLE_HYDE = flags["hyde"]
            ENABLE_HYBRID = flags["hybrid"]
            ENABLE_RERANKING = flags["rerank"]

            ENABLE_SUMMARIZATION = flags["summarize"]
            SUMMARIZATION_TYPE = flags["summ_type"]

            ENABLE_REPACKING = flags["repack"]
            REPACK_STRATEGY = flags["repack_strat"]

            print(f"\n{'=' * 70}")
            print(f"Mode: {mode_name}")
            print(
                f"  HyDE={ENABLE_HYDE}, "
                f"Hybrid={ENABLE_HYBRID}, "
                f"Rerank={ENABLE_RERANKING}"
            )
            print(
                f"  Summarize={ENABLE_SUMMARIZATION} "
                f"({SUMMARIZATION_TYPE if ENABLE_SUMMARIZATION else '-'})"
            )
            print(
                f"  Repack={ENABLE_REPACKING} "
                f"({REPACK_STRATEGY if ENABLE_REPACKING else '-'})"
            )
            print(f"{'=' * 70}")

            df = evaluate_mode(
                ragbench_by_domain=ragbench_by_domain,
                domain_name=domain_name,
                dataset_name=dataset_name,
                llm_client=llm_client,
                top_k=top_k,
                max_samples=max_samples,
                split=split,
                sleep_sec=sleep_sec
            )

            mode_dfs[mode_name] = df
            ablation[mode_name] = compute_metrics(df)

            print(f"{mode_name}: {len(df)} samples evaluated")

            # Useful judge diagnostics
            if not df.empty:
                invalid_relevant = df["invalid_relevant_count"].sum()
                invalid_utilized = df["invalid_utilized_count"].sum()
                invalid_supporting = df["invalid_supporting_count"].sum()

                print(
                    f"  Judge invalid keys cleaned: "
                    f"relevant={invalid_relevant}, "
                    f"utilized={invalid_utilized}, "
                    f"supporting={invalid_supporting}"
                )

    finally:
        # Restore previous global flag state
        ENABLE_HYDE = old_flags["ENABLE_HYDE"]
        ENABLE_HYBRID = old_flags["ENABLE_HYBRID"]
        ENABLE_RERANKING = old_flags["ENABLE_RERANKING"]

        ENABLE_SUMMARIZATION = old_flags["ENABLE_SUMMARIZATION"]
        SUMMARIZATION_TYPE = old_flags["SUMMARIZATION_TYPE"]

        ENABLE_REPACKING = old_flags["ENABLE_REPACKING"]
        REPACK_STRATEGY = old_flags["REPACK_STRATEGY"]

    return ablation, mode_dfs
# # ── Run 6-mode ablation ────────────────────────────────────────
# def run_ablation(
#     ragbench_by_domain,
#     domain_name,
#     dataset_name,
#     llm_client,
#     top_k=3,
#     max_samples=None,
#     split="test",
#     sleep_sec=1.0
# ):
#     global ENABLE_HYDE, ENABLE_HYBRID, ENABLE_RERANKING
#     global ENABLE_SUMMARIZATION, SUMMARIZATION_TYPE
#     global ENABLE_REPACKING, REPACK_STRATEGY

#     modes = [
#         (
#             "Dense (baseline)",
#             {
#                 "hyde": False,
#                 "hybrid": False,
#                 "rerank": False,
#                 "summarize": False,
#                 "summ_type": "recomp",
#                 "repack": False,
#                 "repack_strat": "forward",
#             },
#         ),

#         (
#             "+ Reranker",
#             {
#                 "hyde": False,
#                 "hybrid": False,
#                 "rerank": True,
#                 "summarize": False,
#                 "summ_type": "recomp",
#                 "repack": False,
#                 "repack_strat": "forward",
#             },
#         ),

#         (
#             "Hybrid + Reranker",
#             {
#                 "hyde": False,
#                 "hybrid": True,
#                 "rerank": True,
#                 "summarize": False,
#                 "summ_type": "recomp",
#                 "repack": False,
#                 "repack_strat": "forward",
#             },
#         ),

#         (
#             "+ Summarization (RECOMP)",
#             {
#                 "hyde": False,
#                 "hybrid": True,
#                 "rerank": True,
#                 "summarize": True,
#                 "summ_type": "recomp",
#                 "repack": False,
#                 "repack_strat": "forward",
#             },
#         ),

#         (
#             "+ Repacking (sides)",
#             {
#                 "hyde": False,
#                 "hybrid": True,
#                 "rerank": True,
#                 "summarize": False,
#                 "summ_type": "recomp",
#                 "repack": True,
#                 "repack_strat": "sides",
#             },
#         ),

#         (
#             "Full pipeline (all features)",
#             {
#                 "hyde": True,
#                 "hybrid": True,
#                 "rerank": True,
#                 "summarize": True,
#                 "summ_type": "recomp",
#                 "repack": True,
#                 "repack_strat": "sides",
#             },
#         ),
#     ]

#     ablation = {}
#     mode_dfs = {}

#     for mode_name, flags in modes:
#         ENABLE_HYDE = flags["hyde"]
#         ENABLE_HYBRID = flags["hybrid"]
#         ENABLE_RERANKING = flags["rerank"]

#         ENABLE_SUMMARIZATION = flags["summarize"]
#         SUMMARIZATION_TYPE = flags["summ_type"]

#         ENABLE_REPACKING = flags["repack"]
#         REPACK_STRATEGY = flags["repack_strat"]

#         print(f"\n{'=' * 70}")
#         print(f"Mode: {mode_name}")
#         print(f"  HyDE={ENABLE_HYDE}, Hybrid={ENABLE_HYBRID}, Rerank={ENABLE_RERANKING}")
#         print(f"  Summarize={ENABLE_SUMMARIZATION} ({SUMMARIZATION_TYPE if ENABLE_SUMMARIZATION else '-'})")
#         print(f"  Repack={ENABLE_REPACKING} ({REPACK_STRATEGY if ENABLE_REPACKING else '-'})")
#         print(f"{'=' * 70}")

#         df = evaluate_mode(
#             ragbench_by_domain=ragbench_by_domain,
#             domain_name=domain_name,
#             dataset_name=dataset_name,
#             llm_client=llm_client,
#             top_k=top_k,
#             max_samples=max_samples,
#             split=split,
#             sleep_sec=sleep_sec
#         )

#         mode_dfs[mode_name] = df
#         ablation[mode_name] = compute_metrics(df)

#         print(f"{mode_name}: {len(df)} samples evaluated")

#         # Useful judge diagnostics
#         if not df.empty:
#             invalid_relevant = df["invalid_relevant_count"].sum()
#             invalid_utilized = df["invalid_utilized_count"].sum()
#             invalid_supporting = df["invalid_supporting_count"].sum()

#             print(
#                 f"  Judge invalid keys cleaned: "
#                 f"relevant={invalid_relevant}, "
#                 f"utilized={invalid_utilized}, "
#                 f"supporting={invalid_supporting}"
#             )

#     return ablation, mode_dfs

In [ ]:
from huggingface_hub import login, HfApi, create_repo
import os

# Login (works in VS Code: env var first, secure prompt fallback)
hf_token = os.environ.get("HF_TOKEN") or get_secret_or_prompt(
    "HF_TOKEN",
    "Enter HuggingFace Token: "
)
login(token=hf_token)

# Config — use a regular dataset repo
HF_USERNAME = "Phani555"
HF_REPO_ID = f"{HF_USERNAME}/IIITH-Cohort26-RAG-Batch37-storage"
HF_REPO_TYPE = "dataset"

# Create repo (idempotent — won't error if it exists)
create_repo(
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    exist_ok=True,
    private=False   # set False for public
)

api = HfApi()

# Verify access
files = api.list_repo_files(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE)
print(f"✅ Connected: {HF_REPO_ID}")
print(f"   Type: {HF_REPO_TYPE}")
print(f"   Current files: {len(files)}")
print(f"   View: https://huggingface.co/datasets/{HF_REPO_ID}")

In [ ]:
# @title HF Ablation Storage: Save / Load / Display / Plot

from huggingface_hub import hf_hub_download, list_repo_files
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from datetime import datetime
import tempfile
import uuid

HF_FOLDER    = "ablations"

api = HfApi()

# ───────────────────────────────────────────────────────────────
# 1) SAVE
# ───────────────────────────────────────────────────────────────
def save_ablation_to_hf(
    ablation,
    domain,
    dataset,
    max_samples=None,
    top_k=3,
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    folder=HF_FOLDER,
    extra_metadata=None
):
    """
    Save ablation results to HF with timestamped filenames.
    Never overwrites existing files.
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_name = f"{domain}_{dataset}_{timestamp}"

    # Collision check
    try:
        existing = list_repo_files(repo_id=repo_id, repo_type=repo_type)
        existing_basenames = {
            os.path.basename(f).replace(".json", "")
            for f in existing
            if f.startswith(f"{folder}/") and f.endswith(".json")
        }
        if base_name in existing_basenames:
            base_name = f"{base_name}_{uuid.uuid4().hex[:6]}"
    except Exception:
        pass

    metadata = {
        "domain": domain,
        "dataset": dataset,
        "max_samples": max_samples,
        "top_k": top_k,
        "timestamp": datetime.now().isoformat(),
    }
    if extra_metadata:
        metadata.update(extra_metadata)

    payload = {"metadata": metadata, "results": ablation}

    with tempfile.TemporaryDirectory() as tmpdir:
        json_path = os.path.join(tmpdir, f"{base_name}.json")
        csv_path  = os.path.join(tmpdir, f"{base_name}.csv")

        with open(json_path, "w") as f:
            json.dump(payload, f, indent=2, default=str)

        df = pd.DataFrame(ablation).T
        df.index.name = "Mode"
        df.to_csv(csv_path)

        api.upload_file(
            path_or_fileobj=json_path,
            path_in_repo=f"{folder}/{base_name}.json",
            repo_id=repo_id,
            repo_type=repo_type,
        )
        api.upload_file(
            path_or_fileobj=csv_path,
            path_in_repo=f"{folder}/{base_name}.csv",
            repo_id=repo_id,
            repo_type=repo_type,
        )

    print(f"✅ Saved: {folder}/{base_name}.json")
    print(f"   View: https://huggingface.co/datasets/{repo_id}/tree/main/{folder}")

    return base_name

# ───────────────────────────────────────────────────────────────
# 2) LOAD
# ───────────────────────────────────────────────────────────────
def list_hf_ablations(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, folder=HF_FOLDER):
    """List all saved ablations on HF."""
    files = list_repo_files(repo_id=repo_id, repo_type=repo_type)
    json_files = sorted([
        f for f in files
        if f.startswith(f"{folder}/") and f.endswith(".json")
    ])

    if not json_files:
        print("No ablations found")
        return []

    print(f"Found {len(json_files)} ablations:\n")
    for f in json_files:
        print(f"  {f}")
    return json_files


def load_ablation_from_hf(filename, repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE):
    """
    Load specific ablation by filename.
    Example:
    load_ablation_from_hf("ablations/Customer_Support_emanual_20260623_161858.json")
    """
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        repo_type=repo_type,
    )

    with open(local_path, "r") as f:
        payload = json.load(f)

    return payload["results"], payload.get("metadata", {})


def _parse_timestamp_from_metadata_or_filename(filename, repo_id, repo_type):
    """
    Try metadata timestamp first.
    Fallback to filename timestamp.
    Returns a datetime object.
    """
    # 1) Try metadata timestamp from JSON
    try:
        local_path = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            repo_type=repo_type,
        )

        with open(local_path, "r") as f:
            payload = json.load(f)

        ts = payload.get("metadata", {}).get("timestamp")

        if ts:
            # Handles ISO format like: 2026-06-23T16:18:58.367810
            return datetime.fromisoformat(ts)

    except Exception as e:
        print(f"  ⚠️ Could not read metadata from {filename}: {e}")

    # 2) Fallback: parse timestamp from filename
    # Expected filename pattern:
    # ablations/Customer_Support_emanual_20260623_161858.json

    base = os.path.basename(filename).replace(".json", "")
    parts = base.split("_")

    # Look for last two parts as date + time
    # Example: 20260623 + 161858
    if len(parts) >= 2:
        date_part = parts[-2]
        time_part = parts[-1]

        try:
            return datetime.strptime(
                f"{date_part}_{time_part}",
                "%Y%m%d_%H%M%S"
            )
        except Exception:
            pass

    # 3) If no timestamp found, return very old date
    return datetime.min


def load_latest_ablation(
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    folder=HF_FOLDER,
    domain=None,
    dataset=None
):
    """
    Load the most recent ablation JSON from HF.

    Optional filters:
    - domain="Customer_Support"
    - dataset="emanual"

    Uses metadata timestamp first, filename timestamp second.
    """

    files = list_repo_files(
        repo_id=repo_id,
        repo_type=repo_type
    )

    json_files = [
        f for f in files
        if f.startswith(f"{folder}/") and f.endswith(".json")
    ]

    # Optional domain filter
    if domain:
        json_files = [
            f for f in json_files
            if os.path.basename(f).startswith(f"{domain}_")
        ]

    # Optional dataset filter
    if dataset:
        json_files = [
            f for f in json_files
            if f"_{dataset}_" in os.path.basename(f)
        ]

    if not json_files:
        raise ValueError(
            f"No ablation JSON files found in {repo_id}/{folder} "
            f"for domain={domain}, dataset={dataset}"
        )

    # Sort by actual timestamp
    files_with_ts = [
        (
            f,
            _parse_timestamp_from_metadata_or_filename(
                f,
                repo_id,
                repo_type
            )
        )
        for f in json_files
    ]

    files_with_ts = sorted(files_with_ts, key=lambda x: x[1])

    latest_file, latest_ts = files_with_ts[-1]

    print(f"Loading latest ablation:")
    print(f"  File      : {latest_file}")
    print(f"  Timestamp : {latest_ts}")

    return load_ablation_from_hf(
        latest_file,
        repo_id=repo_id,
        repo_type=repo_type
    )


# ───────────────────────────────────────────────────────────────
# 3) DISPLAY
# ───────────────────────────────────────────────────────────────
def display_ablation(ablation, metadata=None):
    """Print ablation results as a table with deltas vs baseline."""
    df = pd.DataFrame(ablation).T
    df.index.name = "Mode"

    print("\n" + "=" * 90)
    print("ABLATION RESULTS")
    print("=" * 90)

    if metadata:
        print(f"Domain: {metadata.get('domain', 'N/A')}")
        print(f"Dataset: {metadata.get('dataset', 'N/A')}")
        print(f"Samples: {metadata.get('max_samples', 'N/A')}")
        print(f"Timestamp: {metadata.get('timestamp', 'N/A')}")
        print("-" * 90)

    print(df.to_string())

    # Delta vs baseline
    modes = list(ablation.keys())
    if len(modes) >= 2:
        baseline = modes[0]
        print("\n" + "-" * 90)
        print(f"DELTA vs {baseline}")
        print("-" * 90)

        for mode in modes[1:]:
            print(f"\n[{mode}]")
            for metric in ablation[baseline]: # Corrected line
                base = ablation[baseline][metric]
                curr = ablation[mode][metric]

                if isinstance(base, (int, float)) and isinstance(curr, (int, float)):
                    if not (np.isnan(base) or np.isnan(curr)):
                        delta = curr - base
                        if "AUROC" in metric:
                            direction = "improved" if delta > 0 else "degraded"
                        else:
                            direction = "improved" if delta < 0 else "degraded"
                        print(f"  {metric}: {delta:+.4f} ({direction})")

    return df

# ───────────────────────────────────────────────────────────────
# 4) PLOT
# ───────────────────────────────────────────────────────────────
def plot_ablation(ablation, metadata=None, save_path=None):
    """Plot ablation results as grouped bars."""
    modes   = list(ablation.keys())
    metrics = list(ablation[modes[0]].keys())

    n_modes = len(modes)
    x = np.arange(len(metrics))
    width = 0.8 / n_modes

    colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974", "#64B5CD"]

    fig, ax = plt.subplots(figsize=(14, 6))

    for i, mode in enumerate(modes):
        vals = [ablation[mode].get(m, 0) for m in metrics]
        offset = (i - n_modes / 2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width,
                      label=mode, color=colors[i % len(colors)])

        for bar in bars:
            h = bar.get_height()
            if h is not None and not np.isnan(h):
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                        f"{h:.2f}", ha="center", va="bottom", fontsize=7)

    ax.set_ylabel("Score")
    title = "RAG Pipeline Ablation"
    if metadata:
        title += f"\n{metadata.get('domain', '')}/{metadata.get('dataset', '')} (n={metadata.get('max_samples', '?')})"
    ax.set_title(title)

    ax.set_xticks(x)
    ax.set_xticklabels(metrics, rotation=15, ha="right")
    ax.legend(loc="upper right", fontsize=8, bbox_to_anchor=(1.0, 1.0))
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Plot saved: {save_path}")

    plt.show()

#Experiments

In [ ]:
#@title Ablation tests


# ── Feature flags ──────────────────────────────────────────────
ENABLE_HYBRID     = True    # dense + BM25
ENABLE_HYDE       = True    # hypothetical document expansion
ENABLE_RERANKING  = True    # MonoT5 reranker
RERANKER_TYPE = "monot5"     # monot5 | tilde

# ── Tunable knobs ──────────────────────────────────────────────
RETRIEVE_TOP_K    = 10      # initial candidates from each retriever
RERANK_TOP_K      = 3       # final docs after reranking
HYBRID_ALPHA      = 0.5     # 0=BM25 only, 1=dense only, 0.5=balanced
MONOT5_MODEL      = "castorini/monot5-base-msmarco-10k"

#Reciprocal Rank Fusion (RRF)
ENABLE_RRF = True

# Standard value used in literature
RRF_K = 60


# ── Prompt strategy feature flag ────────────────────────────────
PROMPT_STRATEGY = "short"   # "short" | "long" | "long_cot"

# ── Repacking feature flag ──────────────────────────────────────
ENABLE_REPACKING = True      # reorder for LLM attention
REPACK_STRATEGY  = "sides"   # "forward" | "reverse" | "sides"

# ── Feature flags ──────────────────────────────────────────────
ENABLE_SUMMARIZATION = True
SUMMARIZATION_TYPE   = "recomp"   # "recomp" | "longllmlingua"



# ── Run ablation ───────────────────────────────────────────────
ablation_cus, mode_dfs_cus = run_ablation(
    ragbench_by_domain=ragbench_by_domain,
    domain_name="Customer_Support",
    dataset_name="emanual",
    llm_client=llm_client,
    top_k=3,
    max_samples=130
)

# ── Save to HF ─────────────────────────────────────────────────
save_ablation_to_hf(
    ablation_cus,
    domain="Customer_Support",
    dataset="emanual",
    max_samples=130,
    top_k=3,
    extra_metadata={
        "rag_model": MODEL_NAME,
        "judge_model": MODEL_NAME_BIG,
    }
)

# ── List saved ablations ───────────────────────────────────────
list_hf_ablations()


# ── Load latest Finance / finqa ablation specifically ──────────
ablation_cus, metadata_cus = load_latest_ablation(
    domain="Customer_Support",
    dataset="emanual"
)

# ── Display ────────────────────────────────────────────────────
display_ablation(ablation_cus, metadata_cus)


# ── Plot ───────────────────────────────────────────────────────
plot_ablation(ablation_cus, metadata_cus)


In [ ]:
#@title Ablation tests


# ── Feature flags ──────────────────────────────────────────────
ENABLE_HYBRID     = True    # dense + BM25
ENABLE_HYDE       = True    # hypothetical document expansion
ENABLE_RERANKING  = True    # MonoT5 reranker
RERANKER_TYPE = "monot5"     # monot5 | tilde

# ── Tunable knobs ──────────────────────────────────────────────
RETRIEVE_TOP_K    = 10      # initial candidates from each retriever
RERANK_TOP_K      = 3       # final docs after reranking
HYBRID_ALPHA      = 0.5     # 0=BM25 only, 1=dense only, 0.5=balanced
MONOT5_MODEL      = "castorini/monot5-base-msmarco-10k"

#Reciprocal Rank Fusion (RRF)
ENABLE_RRF = True

# Standard value used in literature
RRF_K = 60


# ── Prompt strategy feature flag ────────────────────────────────
PROMPT_STRATEGY = "short"   # "short" | "long" | "long_cot"

# ── Repacking feature flag ──────────────────────────────────────
ENABLE_REPACKING = True      # reorder for LLM attention
REPACK_STRATEGY  = "sides"   # "forward" | "reverse" | "sides"

# ── Feature flags ──────────────────────────────────────────────
ENABLE_SUMMARIZATION = True
SUMMARIZATION_TYPE   = "recomp"   # "recomp" | "longllmlingua"



# ── Run ablation ───────────────────────────────────────────────
ablation_cus, mode_dfs_cus = run_ablation(
    ragbench_by_domain=ragbench_by_domain,
    domain_name="Customer_Support",
    dataset_name="delucionqa",
    llm_client=llm_client,
    top_k=3,
    max_samples=180
)

# ── Save to HF ─────────────────────────────────────────────────
save_ablation_to_hf(
    ablation_cus,
    domain="Customer_Support",
    dataset="delucionqa",
    max_samples=180,
    top_k=3,
    extra_metadata={
        "rag_model": MODEL_NAME,
        "judge_model": MODEL_NAME_BIG,
    }
)

# ── List saved ablations ───────────────────────────────────────
list_hf_ablations()


# ── Load latest Finance / finqa ablation specifically ──────────
ablation_cus, metadata_cus = load_latest_ablation(
    domain="Customer_Support",
    dataset="delucionqa"
)

# ── Display ────────────────────────────────────────────────────
display_ablation(ablation_cus, metadata_cus)


# ── Plot ───────────────────────────────────────────────────────
plot_ablation(ablation_cus, metadata_cus)


In [ ]:
#@title Ablation tests


# ── Feature flags ──────────────────────────────────────────────
ENABLE_HYBRID     = True    # dense + BM25
ENABLE_HYDE       = True    # hypothetical document expansion
ENABLE_RERANKING  = True    # MonoT5 reranker
RERANKER_TYPE = "monot5"     # monot5 | tilde

# ── Tunable knobs ──────────────────────────────────────────────
RETRIEVE_TOP_K    = 10      # initial candidates from each retriever
RERANK_TOP_K      = 3       # final docs after reranking
HYBRID_ALPHA      = 0.5     # 0=BM25 only, 1=dense only, 0.5=balanced
MONOT5_MODEL      = "castorini/monot5-base-msmarco-10k"

#Reciprocal Rank Fusion (RRF)
ENABLE_RRF = True

# Standard value used in literature
RRF_K = 60


# ── Prompt strategy feature flag ────────────────────────────────
PROMPT_STRATEGY = "short"   # "short" | "long" | "long_cot"

# ── Repacking feature flag ──────────────────────────────────────
ENABLE_REPACKING = True      # reorder for LLM attention
REPACK_STRATEGY  = "sides"   # "forward" | "reverse" | "sides"

# ── Feature flags ──────────────────────────────────────────────
ENABLE_SUMMARIZATION = True
SUMMARIZATION_TYPE   = "recomp"   # "recomp" | "longllmlingua"



# ── Run ablation ───────────────────────────────────────────────
ablation_cus, mode_dfs_cus = run_ablation(
    ragbench_by_domain=ragbench_by_domain,
    domain_name="Customer_Support",
    dataset_name="techqa",
    llm_client=llm_client,
    top_k=3,
    max_samples=200
)

# ── Save to HF ─────────────────────────────────────────────────
save_ablation_to_hf(
    ablation_cus,
    domain="Customer_Support",
    dataset="techqa",
    max_samples=200,
    top_k=3,
    extra_metadata={
        "rag_model": MODEL_NAME,
        "judge_model": MODEL_NAME_BIG,
    }
)

# ── List saved ablations ───────────────────────────────────────
list_hf_ablations()


# ── Load latest Finance / finqa ablation specifically ──────────
ablation_cus, metadata_cus = load_latest_ablation(
    domain="Customer_Support",
    dataset="techqa"
)

# ── Display ────────────────────────────────────────────────────
display_ablation(ablation_cus, metadata_cus)


# ── Plot ───────────────────────────────────────────────────────
plot_ablation(ablation_cus, metadata_cus)


In [ ]:
ablation, metadata = load_ablation_from_hf(
    "ablations/Customer_Support_delucionqa_20260623_165044.json"
)

display_ablation(ablation, metadata)
plot_ablation(ablation, metadata)

In [ ]:
ablation, metadata = load_ablation_from_hf(
    "ablations/Finance_finqa_20260623_171855.json"
)

display_ablation(ablation, metadata)
plot_ablation(ablation, metadata)

In [ ]:
ablation, metadata = load_ablation_from_hf(
    "ablations/Finance_finqa_20260623_171855.json"
)

display_ablation(ablation, metadata)
plot_ablation(ablation, metadata)

In [ ]:
#@title Test Results
ablation, metadata = load_ablation_from_hf(
    "ablations/Customer_Support_emanual_20260623_161858.json"
)

display_ablation(ablation, metadata)
plot_ablation(ablation, metadata)



#Full evaluation code